# BIA Image Captioner: training notebook (Kaggle & Colab)

Train the image-captioning model with a **free Kaggle or Colab GPU**, on
synthetic shapes, Flickr8k, Flickr30k or COCO. The notebook is **self-contained**: section 2 writes the
`bia_captioner` package (identical to the repo's `src/`), and training goes
through the same `train()` function as the `cap-train` command.

| | Kaggle | Colab |
|---|---|---|
| GPU | *Settings → Accelerator → GPU T4 x2* (or P100) | *Runtime → Change runtime type → T4 GPU* |
| Internet | *Settings → Internet → On* (to download data) | on by default |
| Session limit | 12 h per session, 30 h GPU / week | ~12 h, can disconnect when idle |
| Checkpoints kept in | `/kaggle/working` (saved with "Save Version") | Google Drive if `USE_GOOGLE_DRIVE = True` |

The first run downloads the dataset (none for shapes, ~4.3 GB for Flickr30k) and the
pretrained encoder weights.

**Sections:** 1. Install · 2. The package · 3. Settings · 4. Data ·
5. Model · 6. Training with a live dashboard · 7. Results · 8. Download

## 1. Install

Kaggle and Colab already have PyTorch, torchvision, transformers and
matplotlib. This only makes sure `datasets` is recent enough.

In [ ]:
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "datasets>=3.6"])
print("ready")

## 2. The package

The cells below write `src/bia_captioner/*.py`, copied from the repo at build
time. Read them in this order: `config` → `tokenizer` → `data` → `model` →
`metrics` → `train`.

In [ ]:
import os

os.makedirs("src/bia_captioner", exist_ok=True)

In [ ]:
%%writefile src/bia_captioner/__init__.py
"""BIA image captioner: image encoder (pretrained ViT/ResNet or BIA-ViT) + Transformer decoder."""

from bia_captioner.config import DataConfig, ModelConfig, TrainConfig

__all__ = ["DataConfig", "ModelConfig", "TrainConfig"]

In [ ]:
%%writefile src/bia_captioner/config.py
"""All hyperparameters live here, grouped by what they control.

Configs are frozen dataclasses: they cannot be changed by accident after
creation, and they are saved inside every checkpoint so a run can be reproduced.
"""

from dataclasses import asdict, dataclass, replace
from pathlib import Path
from typing import Any

# ---------------------------------------------------------------------------
# Image encoders (see model.py). Every one except "bia-vit" can start from
# pretrained weights or from scratch (ModelConfig.pretrained).
# ---------------------------------------------------------------------------
BIA_VIT = "bia-vit"  # the project's own Vision Transformer, always from scratch
TORCHVISION_VITS = ("vit_b_16", "vit_b_32", "vit_l_16", "vit_l_32", "vit_h_14")
TORCHVISION_RESNETS = ("resnet18", "resnet34", "resnet50", "resnet101", "resnet152")
# Any Hugging Face vision backbone: "hf:<model id>", for example
#   hf:google/vit-base-patch16-224-in21k   ViT-B/16 pretrained on ImageNet-21k (14M images)
#   hf:facebook/dinov2-base                DINOv2 ViT-B/14, self-supervised on 142M images
#   hf:openai/clip-vit-base-patch16        CLIP ViT-B/16 image tower, 400M image-text pairs
#   hf:microsoft/resnet-50                 ResNet-50 (ImageNet-1k)
HF_PREFIX = "hf:"


def is_valid_encoder(name: str) -> bool:
    return (
        name == BIA_VIT
        or name in TORCHVISION_VITS
        or name in TORCHVISION_RESNETS
        or (name.startswith(HF_PREFIX) and len(name) > len(HF_PREFIX))
    )


# ---------------------------------------------------------------------------
# Datasets. A preset fills DataConfig's dataset fields; see docs/datasets.md.
# ---------------------------------------------------------------------------
SYNTHETIC = "synthetic"  # generated on the fly, no download (see data.SyntheticShapes)
DATASET_PRESETS: dict[str, dict[str, Any]] = {
    # 8k images of colored shapes, captions fully determined by the image.
    # The quickest proof that the architecture learns from pixels (minutes on a CPU).
    "shapes": {"dataset_id": SYNTHETIC, "caption_columns": ("caption",), "group_column": None},
    # "The CIFAR-10 of captioning": 6 552 high-resolution flower photos, one short
    # caption each (written by BLIP, so simple and consistent), ~0.3 GB. Learns in
    # minutes on a GPU, and the pictures are nice to show. One split only:
    # validation / test images are a seeded random 5% / 10% of it.
    "flowers": {
        "dataset_id": "pranked03/flowers-blip-captions",
        "caption_columns": ("text",),
        "group_column": None,
        "holdout": (0.05, 0.10),
    },
    # CUB-200 birds: 5 994 photos of 200 species, ~10 human captions each about
    # colors and body parts ("this bird has a red crown and a short black bill"),
    # ~0.6 GB. One split only: validation / test are a seeded random 8% / 8%.
    "birds": {
        "dataset_id": "Multimodal-Fatima/CUB_train",
        "caption_columns": ("description",),
        "caption_separator": "\n",
        "group_column": None,
        "holdout": (0.08, 0.08),
    },
    # 8 091 everyday photos, 5 captions each, 6k/1k/1k split, ~0.3 GB.
    "flickr8k": {
        "dataset_id": "jxie/flickr8k",
        "caption_columns": tuple(f"caption_{i}" for i in range(5)),
        "group_column": None,
    },
    # 31 014 photos, 5 captions each, Karpathy split 29k/1k/1k, ~4.3 GB.
    "flickr30k": {
        "dataset_id": "AnyModal/flickr30k",
        "caption_columns": ("original_alt_text",),
        "group_column": None,
    },
    # MS-COCO Karpathy split: 113k/5k/5k images, one caption per row (grouped by
    # image id into ~5 captions per image), ~21 GB. The standard captioning benchmark.
    "coco": {
        "dataset_id": "jxie/coco_captions",
        "caption_columns": ("caption",),
        "group_column": "cocoid",
    },
}
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


@dataclass(frozen=True, slots=True)
class DataConfig:
    """Where the data comes from and how it is turned into tensors."""

    dataset_id: str = "AnyModal/flickr30k"  # Hugging Face dataset, or "synthetic"
    image_column: str = "image"
    # One column holding a list of captions, or several columns with one caption each.
    caption_columns: tuple[str, ...] = ("original_alt_text",)
    # A caption cell holding several captions joined by this string (e.g. "\n").
    caption_separator: str | None = None
    # Rows sharing this column's value are one image (datasets with one caption per row).
    group_column: str | None = None
    # For datasets with a single split: (validation, test) fractions carved out of
    # `train_split` with a fixed random order; the splits are then named
    # "train", "validation" and "test" as usual.
    holdout: tuple[float, float] | None = None
    tokenizer_id: str = "openai-community/gpt2"
    cache_dir: Path | None = None  # Hugging Face cache, None = library default
    train_split: str = "train"
    val_split: str = "validation"  # keep "test" untouched for the final evaluation
    image_size: int = 224
    # Normalisation the encoder expects (ImageNet for torchvision; read from the
    # Hugging Face image processor for "hf:" encoders, see train.py).
    image_mean: tuple[float, float, float] = IMAGENET_MEAN
    image_std: tuple[float, float, float] = IMAGENET_STD
    # Includes BOS and EOS. Flickr30k captions average 16 GPT-2 tokens; 64 fits 99.96%.
    max_tokens: int = 64

    def __post_init__(self) -> None:
        # Checkpoints store tuples as lists; turn them back into tuples.
        for name in ("caption_columns", "image_mean", "image_std", "holdout"):
            if getattr(self, name) is not None:
                object.__setattr__(self, name, tuple(getattr(self, name)))
        if not self.caption_columns:
            raise ValueError("caption_columns must name at least one column")
        if self.holdout is not None and not (
            len(self.holdout) == 2
            and all(0 < f < 1 for f in self.holdout)
            and sum(self.holdout) < 1
        ):
            raise ValueError("holdout must be two fractions in (0, 1) summing to less than 1")
        if self.image_size <= 0:
            raise ValueError("image_size must be positive")
        if self.max_tokens < 2:
            raise ValueError("max_tokens must be at least 2 (room for BOS and EOS)")


def data_config_for(dataset: str, **overrides: Any) -> DataConfig:
    """DataConfig for a preset name (see DATASET_PRESETS)."""
    if dataset not in DATASET_PRESETS:
        raise ValueError(f"unknown dataset {dataset!r}; choose from {list(DATASET_PRESETS)}")
    return replace(DataConfig(), **{**DATASET_PRESETS[dataset], **overrides})


@dataclass(frozen=True, slots=True)
class ModelConfig:
    """Shape of the encoder-decoder network."""

    vocab_size: int
    pad_token_id: int
    bos_token_id: int
    eos_token_id: int
    encoder: str = "vit_b_16"  # see the encoder list at the top of this file
    pretrained: bool = True  # start the encoder from pretrained weights (not for bia-vit)
    d_model: int = 512
    attention_heads: int = 8
    decoder_layers: int = 6
    ffn_mult: int = 4  # feed-forward hidden size = ffn_mult * d_model
    max_tokens: int = 64
    dropout: float = 0.1
    # Only used by the from-scratch "bia-vit" encoder (its width is d_model).
    patch_size: int = 16
    encoder_layers: int = 6

    def __post_init__(self) -> None:
        if self.vocab_size <= 0:
            raise ValueError("vocab_size must be positive")
        if not is_valid_encoder(self.encoder):
            raise ValueError(
                f"unknown encoder {self.encoder!r}: use {BIA_VIT!r}, one of "
                f"{TORCHVISION_VITS + TORCHVISION_RESNETS} or 'hf:<model id>'"
            )
        if self.d_model % self.attention_heads:
            raise ValueError("d_model must be divisible by attention_heads")
        if not 0.0 <= self.dropout < 1.0:
            raise ValueError("dropout must be in [0, 1)")
        if self.pad_token_id in (self.bos_token_id, self.eos_token_id):
            # If pad == eos, the loss ignores EOS and the model never learns to stop.
            raise ValueError("pad_token_id must differ from bos/eos token ids")

    @property
    def encoder_is_pretrained(self) -> bool:
        """True when the encoder starts from pretrained weights (bia-vit never does)."""
        return self.pretrained and self.encoder != BIA_VIT


@dataclass(frozen=True, slots=True)
class TrainConfig:
    """Optimisation and bookkeeping settings for one training run."""

    output_dir: Path = Path("checkpoints/run")
    seed: int = 42
    epochs: int = 15
    batch_size: int = 64
    grad_accum_steps: int = 1  # effective batch = batch_size * grad_accum_steps
    lr: float = 3e-4  # decoder learning rate
    # Encoder learning rate. None = 1e-5 for a pretrained encoder (it is already
    # good, a large LR would destroy it) and = lr for an encoder trained from scratch.
    encoder_lr: float | None = None
    freeze_encoder: bool = False  # True = train only the decoder (much faster)
    weight_decay: float = 0.05
    warmup_steps: int = 500
    grad_clip: float = 1.0
    label_smoothing: float = 0.1
    augment: bool = True  # random crop + colour jitter on training images
    precision: str = "auto"  # "auto" | "fp32" | "bf16" | "fp16"
    num_workers: int = 4
    train_limit: int | None = None  # use only N images (tiny overfit runs)
    val_limit: int | None = None
    log_every: int = 50  # optimizer steps between log lines
    eval_every: int = 500  # optimizer steps between validations
    bleu_samples: int = 500  # val images captioned (greedy) for BLEU at each validation, 0 = off
    device: str = "auto"  # "auto" | "cpu" | "cuda" | "mps"
    resume: Path | None = None  # checkpoint to continue from

    def __post_init__(self) -> None:
        if self.batch_size <= 0 or self.grad_accum_steps <= 0:
            raise ValueError("batch_size and grad_accum_steps must be positive")
        if self.epochs <= 0:
            raise ValueError("epochs must be positive")
        if self.precision not in {"auto", "fp32", "bf16", "fp16"}:
            raise ValueError("precision must be auto, fp32, bf16 or fp16")
        if self.encoder_lr is not None and self.encoder_lr < 0:
            raise ValueError("encoder_lr must be >= 0")


def resolve_encoder_lr(train_config: TrainConfig, model_config: ModelConfig) -> float:
    """The encoder LR to use: explicit value, or a default that depends on its init."""
    if train_config.encoder_lr is not None:
        return train_config.encoder_lr
    return 1e-5 if model_config.encoder_is_pretrained else train_config.lr


def config_to_dict(config: Any) -> dict[str, Any]:
    """Turn a config into plain types so it can be stored in a checkpoint."""
    return {k: str(v) if isinstance(v, Path) else v for k, v in asdict(config).items()}


__all__ = [
    "BIA_VIT",
    "DATASET_PRESETS",
    "HF_PREFIX",
    "IMAGENET_MEAN",
    "IMAGENET_STD",
    "SYNTHETIC",
    "TORCHVISION_RESNETS",
    "TORCHVISION_VITS",
    "DataConfig",
    "ModelConfig",
    "TrainConfig",
    "config_to_dict",
    "data_config_for",
    "is_valid_encoder",
    "resolve_encoder_lr",
]

In [ ]:
%%writefile src/bia_captioner/tokenizer.py
"""Tokenizer setup shared by the dataset, the model and generation.

We reuse GPT-2's byte-level BPE vocabulary but give it a *dedicated* padding
token. GPT-2 has no pad token, and the common trick `pad = eos` is harmful
here: the loss ignores padding, so it would also ignore EOS and the model
would never learn when a caption ends.

Every sequence looks like:  <|endoftext|> caption tokens ... <|endoftext|>
                             (BOS)                            (EOS)
"""

from pathlib import Path

from transformers import AutoTokenizer, PreTrainedTokenizerBase

PAD_TOKEN = "<|pad|>"
IGNORE_INDEX = -100  # label value that cross-entropy skips (used for padding)


def build_tokenizer(tokenizer_id: str, cache_dir: Path | None = None) -> PreTrainedTokenizerBase:
    """Load the base tokenizer and add a dedicated pad token."""
    tokenizer = AutoTokenizer.from_pretrained(
        tokenizer_id, cache_dir=str(cache_dir) if cache_dir else None
    )
    if tokenizer.pad_token is None or tokenizer.pad_token_id == tokenizer.eos_token_id:
        tokenizer.add_special_tokens({"pad_token": PAD_TOKEN})
    if tokenizer.bos_token is None:
        tokenizer.bos_token = tokenizer.eos_token  # GPT-2 style: same token starts and ends
    # We truncate ourselves in encode_caption, so silence the "sequence too long" warning.
    tokenizer.model_max_length = int(1e9)
    return tokenizer


def clean_caption(caption: str) -> str:
    """Normalise whitespace; the model learns the casing and punctuation as written."""
    return " ".join(caption.split())


def encode_caption(tokenizer: PreTrainedTokenizerBase, caption: str, max_tokens: int) -> list[int]:
    """Turn one caption into [BOS, ...caption..., EOS], truncated to max_tokens.

    When the caption is too long we keep the start and drop EOS, so the model
    is never taught to stop in the middle of a sentence.
    """
    body = tokenizer.encode(clean_caption(caption), add_special_tokens=False)
    ids = [tokenizer.bos_token_id, *body, tokenizer.eos_token_id]
    return ids[:max_tokens]


def vocab_size(tokenizer: PreTrainedTokenizerBase) -> int:
    """Size including added tokens (`tokenizer.vocab_size` excludes them)."""
    return len(tokenizer)


__all__ = [
    "IGNORE_INDEX",
    "PAD_TOKEN",
    "build_tokenizer",
    "clean_caption",
    "encode_caption",
    "vocab_size",
]

In [ ]:
%%writefile src/bia_captioner/data.py
"""Datasets, image transforms and batch collation.

Two kinds of data, same interface (one item per image):

- `CaptionDataset`: any Hugging Face image-caption dataset (Flickr8k, Flickr30k,
  COCO, ...). Captions can be one list column, several string columns, or one
  caption per row grouped by an image id.
- `SyntheticShapes`: images of colored shapes generated on the fly, whose
  caption is fully determined by the picture. No download; the fastest way to
  show that the model really reads the image.

Training gives each image a *random* one of its captions, so every epoch goes
through each image once (encoder cost) while all captions get used. Validation
always uses the first caption, so the loss is comparable over time; BLEU
compares with all of them (see `references`).

The datasets return tensors of *different* lengths; the collator pads each
batch only up to its longest caption (dynamic padding), which saves compute.
"""

import random
from collections.abc import Callable
from dataclasses import dataclass
from typing import Any

import numpy as np
import torch
from datasets import load_dataset
from PIL import Image, ImageDraw
from torch.utils.data import Dataset
from torchvision.transforms import v2
from transformers import PreTrainedTokenizerBase

from bia_captioner.config import IMAGENET_MEAN, IMAGENET_STD, SYNTHETIC, DataConfig
from bia_captioner.tokenizer import IGNORE_INDEX, clean_caption, encode_caption


def build_image_transform(
    image_size: int,
    train: bool,
    mean: tuple[float, ...] = IMAGENET_MEAN,
    std: tuple[float, ...] = IMAGENET_STD,
) -> v2.Compose:
    """PIL image -> normalised float tensor [3, H, W].

    `train=True` adds a mild random crop and colour jitter. No horizontal flip:
    it would make captions with "left" / "right" wrong.
    """
    if train:
        steps: list[Any] = [
            v2.RandomResizedCrop(image_size, scale=(0.75, 1.0), antialias=True),
            v2.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        ]
    else:
        steps = [v2.Resize((image_size, image_size), antialias=True)]
    steps += [
        v2.ToImage(),  # PIL -> uint8 tensor [3, H, W]
        v2.ToDtype(torch.float32, scale=True),  # uint8 [0, 255] -> float [0, 1]
        v2.Normalize(mean, std),  # what the encoder was trained with
    ]
    return v2.Compose(steps)


def to_rgb_image(image: Any) -> Image.Image:
    """Accept a PIL image or a numpy array and return an RGB PIL image."""
    if isinstance(image, Image.Image):
        return image.convert("RGB")
    array = np.asarray(image)
    if np.issubdtype(array.dtype, np.floating):
        # Float images are assumed to be in [0, 1]; PIL needs uint8 in [0, 255].
        array = (np.clip(array, 0.0, 1.0) * 255).round().astype(np.uint8)
    return Image.fromarray(array).convert("RGB")


def as_caption_list(value: Any, separator: str | None = None) -> list[str]:
    """A caption cell may hold one string, several joined by `separator`, or a list."""
    captions = [value] if isinstance(value, str) else list(value)
    if separator is not None:
        captions = [part for c in captions for part in c.split(separator)]
    return [clean_caption(c) for c in captions if c and c.strip()]


HOLDOUT_SEED = 0  # fixed: the same images are held out on every machine and every run


def holdout_indices(n: int, holdout: tuple[float, float], split: str) -> list[int]:
    """Rows of a single-split dataset that belong to "train", "validation" or "test".

    A fixed random permutation (rows are often sorted by class), then the first
    `holdout[0]` fraction is validation, the next `holdout[1]` is test, the rest train.
    """
    order = np.random.RandomState(HOLDOUT_SEED).permutation(n).tolist()
    n_val, n_test = round(n * holdout[0]), round(n * holdout[1])
    parts = {
        "validation": order[:n_val],
        "test": order[n_val : n_val + n_test],
        "train": order[n_val + n_test :],
    }
    if split not in parts:
        raise ValueError(f"with a holdout the splits are {list(parts)}, not {split!r}")
    return sorted(parts[split])  # sorted: faster reads from the underlying files


class _CaptionedImages(Dataset[dict[str, torch.Tensor]]):
    """Shared logic: subclasses provide `__len__`, `image(i)` and `self.captions`."""

    captions: list[list[str]]

    def __init__(
        self, config: DataConfig, tokenizer: PreTrainedTokenizerBase, augment: bool
    ) -> None:
        self.tokenizer = tokenizer
        self.max_tokens = config.max_tokens
        self.train = augment  # also picks a random caption per visit
        self.transform = build_image_transform(
            config.image_size, augment, config.image_mean, config.image_std
        )

    def image(self, index: int) -> Image.Image:
        raise NotImplementedError

    def references(self, index: int) -> list[str]:
        """All human captions of one image (the references for BLEU)."""
        return self.captions[index]

    def __getitem__(self, index: int) -> dict[str, torch.Tensor]:
        captions = self.captions[index]
        caption = random.choice(captions) if self.train else captions[0]
        ids = encode_caption(self.tokenizer, caption, self.max_tokens)
        return {
            "pixel_values": self.transform(self.image(index)),
            "input_ids": torch.tensor(ids, dtype=torch.long),
            "index": torch.tensor(index, dtype=torch.long),  # to look up references
        }


class CaptionDataset(_CaptionedImages):
    """Image / caption pairs from a Hugging Face dataset, one item per image."""

    def __init__(
        self,
        config: DataConfig,
        tokenizer: PreTrainedTokenizerBase,
        split: str,
        limit: int | None = None,
        train: bool = False,
        dataset_loader: Callable[..., Any] = load_dataset,
    ) -> None:
        if limit is not None and limit <= 0:
            raise ValueError("limit must be positive")
        super().__init__(config, tokenizer, augment=train)
        self.image_column = config.image_column

        # The loader is injectable so tests can run without the network.
        cache_dir = str(config.cache_dir) if config.cache_dir else None
        if config.holdout is None:
            self.samples = dataset_loader(config.dataset_id, split=split, cache_dir=cache_dir)
        else:  # one split on disk, cut into train / validation / test here
            full = dataset_loader(config.dataset_id, split=config.train_split, cache_dir=cache_dir)
            self.samples = full.select(holdout_indices(len(full), config.holdout, split))
        if limit is not None and config.group_column is None:
            # Keep only the first `limit` images (handy for quick overfit runs).
            self.samples = self.samples.select(range(min(limit, len(self.samples))))

        # Read the caption columns once: cheap (no image decoding) and needed for BLEU.
        columns = [list(self.samples[name]) for name in config.caption_columns]
        separator = config.caption_separator
        per_row = [
            [c for cell in cells for c in as_caption_list(cell, separator)]
            for cells in zip(*columns)
        ]
        if config.group_column is None:
            self.rows = list(range(len(per_row)))
            self.captions = per_row
        else:
            # One caption per row: gather the rows of each image (first row gives the image).
            groups: dict[Any, list[int]] = {}
            for row, key in enumerate(self.samples[config.group_column]):
                groups.setdefault(key, []).append(row)
            self.rows = [rows[0] for rows in groups.values()]
            self.captions = [[c for r in rows for c in per_row[r]] for rows in groups.values()]
            if limit is not None:
                self.rows, self.captions = self.rows[:limit], self.captions[:limit]
        if any(not captions for captions in self.captions):
            raise ValueError("found an image without any caption")

    def __len__(self) -> int:
        return len(self.rows)

    def image(self, index: int) -> Image.Image:
        return to_rgb_image(self.samples[self.rows[index]][self.image_column])


# ---------------------------------------------------------------------------
# Synthetic shapes
# ---------------------------------------------------------------------------

SHAPE_COLORS = {
    "red": (220, 40, 40),
    "green": (40, 160, 60),
    "blue": (40, 80, 220),
    "yellow": (230, 190, 20),
}
SHAPE_KINDS = ("circle", "square", "triangle")
SHAPE_SIZES = {"small": 0.45, "big": 0.85}  # fraction of the object's cell
SYNTHETIC_SPLITS = {"train": (0, 8000), "validation": (1, 500), "test": (2, 500)}  # (seed, size)


def render_shapes(rng: random.Random, canvas: int = 128) -> tuple[Image.Image, str]:
    """Draw one or two random shapes and return (image, caption).

    "a big red circle" or
    "a small blue square on the left and a big yellow triangle on the right".
    """
    count = rng.choice((1, 2))
    image = Image.new("RGB", (canvas, canvas), "white")
    draw = ImageDraw.Draw(image)
    cell = canvas if count == 1 else canvas // 2
    phrases = []
    for slot in range(count):
        size_name, color, kind = (
            rng.choice(list(SHAPE_SIZES)),
            rng.choice(list(SHAPE_COLORS)),
            rng.choice(SHAPE_KINDS),
        )
        side = SHAPE_SIZES[size_name] * cell * 0.8
        # Random position inside the object's cell (left / right half, or the whole image).
        x0 = slot * cell + rng.uniform(0, cell - side)
        y0 = rng.uniform(canvas / 2 - cell / 2, canvas / 2 + cell / 2 - side)
        box = (x0, y0, x0 + side, y0 + side)
        fill = SHAPE_COLORS[color]
        if kind == "circle":
            draw.ellipse(box, fill=fill)
        elif kind == "square":
            draw.rectangle(box, fill=fill)
        else:
            draw.polygon([(x0 + side / 2, y0), (x0, y0 + side), (x0 + side, y0 + side)], fill=fill)
        phrases.append(f"a {size_name} {color} {kind}")
    if count == 1:
        return image, phrases[0]
    return image, f"{phrases[0]} on the left and {phrases[1]} on the right"


class SyntheticShapes(_CaptionedImages):
    """Generated colored-shape images; train / validation / test use different seeds.

    The caption is fully determined by the image, so a model that ignores the
    image cannot do better than guessing, while a working model reaches ~100%.
    """

    def __init__(
        self,
        config: DataConfig,
        tokenizer: PreTrainedTokenizerBase,
        split: str,
        limit: int | None = None,
        train: bool = False,
    ) -> None:
        if split not in SYNTHETIC_SPLITS:
            raise ValueError(f"synthetic splits are {list(SYNTHETIC_SPLITS)}")
        # No augmentation: a random crop could cut a shape off or move it across halves.
        super().__init__(config, tokenizer, augment=False)
        self.seed, size = SYNTHETIC_SPLITS[split]
        self.size = size if limit is None else min(limit, size)
        self.captions = [[render_shapes(self._rng(i))[1]] for i in range(self.size)]

    def _rng(self, index: int) -> random.Random:
        return random.Random(self.seed * 1_000_003 + index)  # same image for the same index

    def __len__(self) -> int:
        return self.size

    def image(self, index: int) -> Image.Image:
        return render_shapes(self._rng(index))[0]


def build_dataset(
    config: DataConfig,
    tokenizer: PreTrainedTokenizerBase,
    split: str,
    limit: int | None = None,
    train: bool = False,
    dataset_loader: Callable[..., Any] | None = None,
) -> Dataset:
    """The dataset named by config.dataset_id ("synthetic" or a Hugging Face id)."""
    if config.dataset_id == SYNTHETIC:
        return SyntheticShapes(config, tokenizer, split, limit=limit, train=train)
    extra = {} if dataset_loader is None else {"dataset_loader": dataset_loader}
    return CaptionDataset(config, tokenizer, split, limit=limit, train=train, **extra)


@dataclass
class CaptionCollator:
    """Stack samples into a batch, padding captions to the longest one.

    Labels are a copy of input_ids with padding set to -100 so the loss
    ignores it. They are *not* shifted here: the model does the shift itself.
    """

    pad_token_id: int

    def __call__(self, batch: list[dict[str, torch.Tensor]]) -> dict[str, torch.Tensor]:
        pixel_values = torch.stack([sample["pixel_values"] for sample in batch])

        sequences = [sample["input_ids"] for sample in batch]
        lengths = torch.tensor([len(seq) for seq in sequences])
        # Right padding: real tokens first, pad tokens after.
        input_ids = torch.nn.utils.rnn.pad_sequence(
            sequences, batch_first=True, padding_value=self.pad_token_id
        )

        # Mark padding by position, not by value, so a real token is never masked.
        is_padding = torch.arange(input_ids.size(1)) >= lengths[:, None]
        labels = input_ids.masked_fill(is_padding, IGNORE_INDEX)

        out = {"pixel_values": pixel_values, "input_ids": input_ids, "labels": labels}
        if "index" in batch[0]:
            out["index"] = torch.stack([sample["index"] for sample in batch])
        return out


__all__ = [
    "IMAGENET_MEAN",
    "IMAGENET_STD",
    "CaptionCollator",
    "CaptionDataset",
    "SyntheticShapes",
    "as_caption_list",
    "build_dataset",
    "build_image_transform",
    "holdout_indices",
    "render_shapes",
    "to_rgb_image",
]

In [ ]:
%%writefile src/bia_captioner/model.py
"""Image -> caption model.

Big picture:
    image --encoder--> N "image tokens" --projection + position + LayerNorm--> memory
    caption tokens --Transformer decoder (cross-attends to memory)--> next-token logits

Image encoders (`ModelConfig.encoder`; all but bia-vit can be pretrained or from scratch):

    "bia-vit"                 BIA Vision Transformer (hand-written), always from scratch
    "vit_b_16" ... "vit_h_14" torchvision Vision Transformers (ImageNet weights)
    "resnet18" ... "resnet152" torchvision ResNets (ImageNet weights)
    "hf:<model id>"           any Hugging Face vision backbone (ViT-21k, DINOv2, CLIP, ...)

The decoder is a hand-written pre-norm Transformer (masked self-attention,
cross-attention to the image tokens, feed-forward), GPT-2 style.
"""

import inspect
import math

import torch
import torchvision
from torch import Tensor, nn
from torch.nn import functional as F

from bia_captioner.config import (
    BIA_VIT,
    HF_PREFIX,
    IMAGENET_MEAN,
    IMAGENET_STD,
    TORCHVISION_VITS,
    ModelConfig,
)
from bia_captioner.tokenizer import IGNORE_INDEX

INIT_STD = 0.02  # GPT-2 style init for every new (non-pretrained) weight
RESNET_STRIDE = 32  # a ResNet shrinks the image 32x before its final pooling


# ---------------------------------------------------------------------------
# Building blocks (shared by the BIA-ViT encoder and the decoder)
# ---------------------------------------------------------------------------


class MultiHeadAttention(nn.Module):
    """Multi-head attention. Self-attention when `context` is None, else cross-attention.

    Uses PyTorch's fused scaled_dot_product_attention (FlashAttention /
    memory-efficient kernels on GPU).
    """

    def __init__(self, d_model: int, heads: int, dropout: float) -> None:
        super().__init__()
        self.heads = heads
        self.dropout = dropout
        self.q_proj = nn.Linear(d_model, d_model)
        self.kv_proj = nn.Linear(d_model, 2 * d_model)  # keys and values in one matmul
        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, x: Tensor, context: Tensor | None = None, causal: bool = False) -> Tensor:
        """x [B, T, d] attends to context [B, S, d] (or to itself) -> [B, T, d]."""
        context = x if context is None else context
        batch, t_len, d_model = x.shape
        s_len = context.shape[1]
        head_dim = d_model // self.heads

        # [B, T, d] -> [B, heads, T, head_dim]
        q = self.q_proj(x).view(batch, t_len, self.heads, head_dim).transpose(1, 2)
        kv = self.kv_proj(context).view(batch, s_len, 2, self.heads, head_dim)
        k, v = kv.permute(2, 0, 3, 1, 4)  # each [B, heads, S, head_dim]

        out = F.scaled_dot_product_attention(
            q, k, v, dropout_p=self.dropout if self.training else 0.0, is_causal=causal
        )
        return self.out_proj(out.transpose(1, 2).reshape(batch, t_len, d_model))


class FeedForward(nn.Module):
    """Position-wise MLP: d -> mult*d -> d with GELU."""

    def __init__(self, d_model: int, mult: int, dropout: float) -> None:
        super().__init__()
        self.fc1 = nn.Linear(d_model, mult * d_model)
        self.fc2 = nn.Linear(mult * d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: Tensor) -> Tensor:
        return self.fc2(self.dropout(F.gelu(self.fc1(x))))


class EncoderBlock(nn.Module):
    """Pre-norm Transformer encoder block: x + attn(norm(x)), then x + mlp(norm(x))."""

    def __init__(self, d_model: int, heads: int, mult: int, dropout: float) -> None:
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, heads, dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.mlp = FeedForward(d_model, mult, dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: Tensor) -> Tensor:
        x = x + self.dropout(self.attn(self.norm1(x)))
        return x + self.dropout(self.mlp(self.norm2(x)))


class DecoderBlock(nn.Module):
    """Pre-norm decoder block: masked self-attention, cross-attention, feed-forward."""

    def __init__(self, d_model: int, heads: int, mult: int, dropout: float) -> None:
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.self_attn = MultiHeadAttention(d_model, heads, dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.cross_attn = MultiHeadAttention(d_model, heads, dropout)
        self.norm3 = nn.LayerNorm(d_model)
        self.mlp = FeedForward(d_model, mult, dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: Tensor, memory: Tensor) -> Tensor:
        # Causal mask: token t only sees tokens 0..t. Right padding comes after
        # every real token, so real tokens never see padding either.
        x = x + self.dropout(self.self_attn(self.norm1(x), causal=True))
        # The memory is already normalised (image_norm), so only x is normed here.
        x = x + self.dropout(self.cross_attn(self.norm2(x), context=memory))
        return x + self.dropout(self.mlp(self.norm3(x)))


# ---------------------------------------------------------------------------
# Image encoders: images [B, 3, H, W] -> features [B, N, C]
# ---------------------------------------------------------------------------


class BIAViT(nn.Module):
    """The BIA Vision Transformer, trained from scratch.

    Patches of `patch_size` pixels -> linear embedding (a strided conv) ->
    learned position -> pre-norm Transformer blocks -> final LayerNorm.
    """

    def __init__(self, config: ModelConfig, image_size: int) -> None:
        super().__init__()
        if image_size % config.patch_size:
            raise ValueError(
                f"image_size {image_size} must be a multiple of patch_size {config.patch_size}"
            )
        d_model = config.d_model
        self.num_tokens = (image_size // config.patch_size) ** 2
        self.out_dim = d_model
        # A conv with kernel = stride = patch size embeds each patch independently.
        self.patch_embedding = nn.Conv2d(
            3, d_model, kernel_size=config.patch_size, stride=config.patch_size
        )
        self.position = nn.Parameter(torch.zeros(1, self.num_tokens, d_model))
        self.dropout = nn.Dropout(config.dropout)
        self.blocks = nn.ModuleList(
            EncoderBlock(d_model, config.attention_heads, config.ffn_mult, config.dropout)
            for _ in range(config.encoder_layers)
        )
        # Pre-norm blocks leave the output un-normalised, so add a final LayerNorm.
        self.norm = nn.LayerNorm(d_model)

    def forward(self, pixel_values: Tensor) -> Tensor:
        x = self.patch_embedding(pixel_values)  # [B, d, H/p, W/p]
        x = x.flatten(2).transpose(1, 2)  # [B, N, d]
        x = self.dropout(x + self.position)
        for block in self.blocks:
            x = block(x)
        return self.norm(x)


class TorchvisionViT(nn.Module):
    """A torchvision Vision Transformer without its classifier: all token features.

    pretrained=True loads the default ImageNet weights, which fix the image size
    (224 for vit_b_16 ... vit_l_32, 518 for vit_h_14).
    """

    def __init__(self, name: str, image_size: int, pretrained: bool) -> None:
        super().__init__()
        builder = getattr(torchvision.models, name)
        if pretrained:
            weights = torchvision.models.get_model_weights(name).DEFAULT
            native = weights.meta["min_size"][0]
            if image_size != native:
                raise ValueError(f"pretrained {name} expects --image-size {native}")
            self.vit = builder(weights=weights)
        else:
            self.vit = builder(weights=None, image_size=image_size)
        if image_size % self.vit.patch_size:
            raise ValueError(f"image_size must be a multiple of {self.vit.patch_size} for {name}")
        self.vit.heads = nn.Identity()  # drop the ImageNet classifier
        self.num_tokens = (image_size // self.vit.patch_size) ** 2 + 1  # patches + class token
        self.out_dim = self.vit.hidden_dim

    def forward(self, pixel_values: Tensor) -> Tensor:
        x = self.vit._process_input(pixel_values)  # patch embedding [B, N, width]
        class_token = self.vit.class_token.expand(x.shape[0], -1, -1)
        x = torch.cat([class_token, x], dim=1)
        return self.vit.encoder(x)  # adds positions, runs the blocks, final LayerNorm


class TorchvisionResNet(nn.Module):
    """A torchvision ResNet without pooling/classifier: one token per grid cell."""

    def __init__(self, name: str, image_size: int, pretrained: bool) -> None:
        super().__init__()
        backbone = torchvision.models.get_model(name, weights="DEFAULT" if pretrained else None)
        self.out_dim = backbone.fc.in_features  # 512 for resnet18/34, 2048 for 50/101/152
        self.body = nn.Sequential(*list(backbone.children())[:-2])  # keep the spatial grid
        self.num_tokens = math.ceil(image_size / RESNET_STRIDE) ** 2  # 224px -> 7x7 = 49

    def forward(self, pixel_values: Tensor) -> Tensor:
        return self.body(pixel_values).flatten(2).transpose(1, 2)  # [B, h*w, C]


class HuggingFaceEncoder(nn.Module):
    """Any Hugging Face vision backbone; its last hidden states are the image tokens.

    CLIP / SigLIP checkpoints contain a text tower too: only the image tower is
    kept. Parameters that never influence the tokens (poolers, projection
    heads) are found with one probe pass and frozen, so DistributedDataParallel
    does not wait for gradients that never come.
    """

    def __init__(self, model_id: str, image_size: int, pretrained: bool) -> None:
        super().__init__()
        from transformers import AutoConfig, AutoModel

        if pretrained:
            model = AutoModel.from_pretrained(model_id)
        else:
            model = AutoModel.from_config(AutoConfig.from_pretrained(model_id))
        self.model = getattr(model, "vision_model", model)  # CLIP / SigLIP: image tower
        config = self.model.config
        self.out_dim = getattr(config, "hidden_size", None) or config.hidden_sizes[-1]
        accepts = inspect.signature(self.model.forward).parameters
        # ViT-style models need this to run at a size other than their native one.
        self.extra = (
            {"interpolate_pos_encoding": True} if "interpolate_pos_encoding" in accepts else {}
        )

        # Probe pass: number of tokens, and which parameters get no gradient.
        tokens = self(torch.zeros(1, 3, image_size, image_size))
        self.num_tokens = tokens.shape[1]
        tokens.float().sum().backward()
        self.unused_parameters = set()
        for name, param in self.named_parameters():
            if param.grad is None:
                self.unused_parameters.add(name)
                param.requires_grad_(False)
            param.grad = None

    def forward(self, pixel_values: Tensor) -> Tensor:
        hidden = self.model(pixel_values=pixel_values, **self.extra).last_hidden_state
        if hidden.dim() == 4:  # convolutional backbone: [B, C, h, w] -> [B, h*w, C]
            hidden = hidden.flatten(2).transpose(1, 2)
        return hidden


def build_encoder(config: ModelConfig, image_size: int, pretrained: bool) -> nn.Module:
    """The image encoder named by config.encoder."""
    if config.encoder == BIA_VIT:
        return BIAViT(config, image_size)  # always from scratch
    if config.encoder.startswith(HF_PREFIX):
        return HuggingFaceEncoder(config.encoder[len(HF_PREFIX) :], image_size, pretrained)
    if config.encoder in TORCHVISION_VITS:
        return TorchvisionViT(config.encoder, image_size, pretrained)
    return TorchvisionResNet(config.encoder, image_size, pretrained)


def encoder_normalization(config: ModelConfig) -> tuple[tuple[float, ...], tuple[float, ...]]:
    """(mean, std) the encoder's inputs must be normalised with.

    torchvision models and BIA-ViT use ImageNet statistics; a Hugging Face
    model brings its own (CLIP and ViT-21k differ from ImageNet).
    """
    if config.encoder.startswith(HF_PREFIX):
        from transformers import AutoImageProcessor

        processor = AutoImageProcessor.from_pretrained(config.encoder[len(HF_PREFIX) :])
        mean, std = getattr(processor, "image_mean", None), getattr(processor, "image_std", None)
        if mean is not None and std is not None:
            return tuple(mean), tuple(std)
    return IMAGENET_MEAN, IMAGENET_STD


# ---------------------------------------------------------------------------
# The captioner
# ---------------------------------------------------------------------------


class ImageCaptioner(nn.Module):
    """Image encoder + autoregressive Transformer decoder."""

    def __init__(
        self, config: ModelConfig, image_size: int = 224, pretrained_encoder: bool | None = None
    ) -> None:
        """pretrained_encoder: None = follow config.pretrained. Loading a checkpoint
        passes False: its weights replace the pretrained ones anyway."""
        super().__init__()
        self.config = config
        if pretrained_encoder is None:
            pretrained_encoder = config.pretrained
        self.encoder_frozen = False
        d_model = config.d_model

        # ---- Image encoder -------------------------------------------------
        self.encoder = build_encoder(config, image_size, pretrained_encoder)
        self.num_image_tokens = self.encoder.num_tokens
        # Map encoder features to the decoder width.
        self.image_projection = nn.Linear(self.encoder.out_dim, d_model)
        # Attention ignores order, so each image token gets a learned "where am I" vector.
        self.image_position = nn.Parameter(torch.zeros(1, self.num_image_tokens, d_model))
        # Normalise the memory so its scale matches what the decoder expects.
        self.image_norm = nn.LayerNorm(d_model)

        # ---- Text decoder --------------------------------------------------
        # padding_idx: the input lookup never sends gradient to the pad row (the tied
        # output head still does: it learns that pad is never the next token).
        self.token_embedding = nn.Embedding(
            config.vocab_size, d_model, padding_idx=config.pad_token_id
        )
        self.position_embedding = nn.Embedding(config.max_tokens, d_model)
        self.embedding_dropout = nn.Dropout(config.dropout)
        self.blocks = nn.ModuleList(
            DecoderBlock(d_model, config.attention_heads, config.ffn_mult, config.dropout)
            for _ in range(config.decoder_layers)
        )
        self.final_norm = nn.LayerNorm(d_model)
        # Output head shares its weights with the token embedding ("weight tying"):
        # fewer parameters and usually a better language model.
        self.output = nn.Linear(d_model, config.vocab_size, bias=False)

        self._init_new_weights()
        self.output.weight = self.token_embedding.weight  # tie after init

    # ------------------------------------------------------------------ init
    def _init_new_weights(self) -> None:
        """N(0, 0.02) for new Linear/Conv/Embedding weights, zero biases.

        Residual output projections get a smaller std (GPT-2): each block adds
        to the residual stream, so this keeps its variance from growing with depth.
        Library encoders (torchvision / Hugging Face) keep their own weights,
        pretrained or freshly initialised by the library; LayerNorms keep (1, 0).
        """
        cfg = self.config
        skip_encoder = not isinstance(self.encoder, BIAViT)
        for name, module in self.named_modules():
            in_encoder = name.startswith("encoder")
            if skip_encoder and in_encoder:
                continue
            if isinstance(module, (nn.Linear, nn.Conv2d, nn.Embedding)):
                std = INIT_STD
                if name.endswith(("out_proj", "fc2")):  # writes into the residual stream
                    depth = cfg.encoder_layers if in_encoder else cfg.decoder_layers
                    std = INIT_STD / math.sqrt(2 * depth)
                nn.init.normal_(module.weight, mean=0.0, std=std)
            if isinstance(module, (nn.Linear, nn.Conv2d)) and module.bias is not None:
                nn.init.zeros_(module.bias)
        nn.init.normal_(self.image_position, mean=0.0, std=INIT_STD)
        if isinstance(self.encoder, BIAViT):
            nn.init.normal_(self.encoder.position, mean=0.0, std=INIT_STD)
        with torch.no_grad():
            self.token_embedding.weight[self.config.pad_token_id].zero_()  # pad starts at zero

    # -------------------------------------------------------------- freezing
    def freeze_encoder(self, frozen: bool = True) -> None:
        """Freeze (or unfreeze) the image encoder.

        A frozen encoder gets no gradients, runs without autograd (faster, less
        memory) and stays in eval mode so BatchNorm/dropout behave as at inference.
        """
        self.encoder_frozen = frozen
        never_trained = getattr(self.encoder, "unused_parameters", set())
        for name, param in self.encoder.named_parameters():
            param.requires_grad_(not frozen and name not in never_trained)
        self.train(self.training)

    def train(self, mode: bool = True) -> "ImageCaptioner":
        super().train(mode)
        if self.encoder_frozen:
            self.encoder.eval()
        return self

    # --------------------------------------------------------------- encoder
    def encode(self, pixel_values: Tensor) -> Tensor:
        """Images [B, 3, H, W] -> memory [B, N, d_model]."""
        with torch.set_grad_enabled(torch.is_grad_enabled() and not self.encoder_frozen):
            features = self.encoder(pixel_values)  # [B, N, C]
        if features.shape[1] != self.num_image_tokens:
            raise ValueError(
                f"encoder produced {features.shape[1]} image tokens but the model was built "
                f"for {self.num_image_tokens}; pass the right image_size to the constructor"
            )
        memory = self.image_projection(features) + self.image_position
        return self.image_norm(memory)

    # --------------------------------------------------------------- decoder
    def decode(self, memory: Tensor, input_ids: Tensor) -> Tensor:
        """Memory + caption tokens [B, T] -> logits [B, T, vocab]."""
        seq_len = input_ids.shape[1]
        if seq_len > self.config.max_tokens:
            raise ValueError(f"sequence length {seq_len} exceeds max_tokens")

        positions = torch.arange(seq_len, device=input_ids.device)
        x = self.token_embedding(input_ids) + self.position_embedding(positions)
        x = self.embedding_dropout(x)
        for block in self.blocks:
            x = block(x, memory)
        return self.output(self.final_norm(x))

    def forward(
        self,
        pixel_values: Tensor,
        input_ids: Tensor,
        labels: Tensor | None = None,
        label_smoothing: float = 0.0,
    ) -> tuple[Tensor, Tensor | None]:
        """Return (logits, loss). Loss is None when no labels are given.

        `labels` is aligned with `input_ids` (same shape); padding is -100.
        We shift inside: the logits at position t predict the label at t + 1.
        """
        logits = self.decode(self.encode(pixel_values), input_ids)
        loss = None
        if labels is not None:
            loss = F.cross_entropy(
                logits[:, :-1].reshape(-1, logits.shape[-1]).float(),
                labels[:, 1:].reshape(-1),
                ignore_index=IGNORE_INDEX,
                label_smoothing=label_smoothing,
            )
        return logits, loss

    # ------------------------------------------------------------ generation
    def _forbid_special_tokens(self, logits: Tensor) -> Tensor:
        """Pad (and BOS, when it is not also EOS) are never real predictions."""
        logits[:, self.config.pad_token_id] = float("-inf")
        if self.config.bos_token_id != self.config.eos_token_id:
            logits[:, self.config.bos_token_id] = float("-inf")
        return logits

    def _max_new_tokens(self, max_new_tokens: int | None) -> int:
        limit = self.config.max_tokens - 1  # BOS already uses one slot
        return limit if max_new_tokens is None else min(max_new_tokens, limit)

    @torch.no_grad()
    def generate(
        self,
        pixel_values: Tensor,
        max_new_tokens: int | None = None,
        temperature: float = 0.0,
        top_k: int | None = None,
        top_p: float | None = None,
    ) -> Tensor:
        """Caption each image, one token at a time (greedy or sampling).

        Returns [B, T] token ids starting with BOS. Rows that already produced
        EOS are filled with pad. temperature=0 means greedy (always the top token).
        Train/eval mode is left to the caller (call model.eval() first).
        """
        cfg = self.config
        memory = self.encode(pixel_values)  # image is encoded once and reused
        batch = pixel_values.shape[0]
        device = pixel_values.device
        tokens = torch.full((batch, 1), cfg.bos_token_id, dtype=torch.long, device=device)
        finished = torch.zeros(batch, dtype=torch.bool, device=device)

        for _ in range(self._max_new_tokens(max_new_tokens)):
            logits = self.decode(memory, tokens)[:, -1].float()  # only the last position matters
            self._forbid_special_tokens(logits)
            next_token = _sample(logits, temperature, top_k, top_p)
            next_token = next_token.masked_fill(finished, cfg.pad_token_id)  # done rows get pad
            tokens = torch.cat([tokens, next_token[:, None]], dim=1)
            finished |= next_token.eq(cfg.eos_token_id)
            if finished.all():
                break
        return tokens

    @torch.no_grad()
    def beam_search(
        self,
        pixel_values: Tensor,
        beam_size: int = 3,
        max_new_tokens: int | None = None,
        length_penalty: float = 0.6,
    ) -> list[tuple[Tensor, float]]:
        """Beam search for ONE image [1, 3, H, W].

        Keeps the `beam_size` most likely partial captions at each step. Scores
        are summed log-probabilities divided by the GNMT length penalty
        ((5 + length) / 6) ** length_penalty, so short captions are not unfairly
        preferred (0 = no normalisation). Returns up to `beam_size`
        (token ids starting with BOS, score) pairs, best first.
        """
        if pixel_values.shape[0] != 1:
            raise ValueError("beam_search takes one image; loop over images for a batch")
        if beam_size < 1:
            raise ValueError("beam_size must be >= 1")
        cfg = self.config
        memory = self.encode(pixel_values)
        beams = torch.full((1, 1), cfg.bos_token_id, dtype=torch.long, device=memory.device)
        scores = torch.zeros(1, device=memory.device)
        finished: list[tuple[list[int], float]] = []

        def normalised(score: float, length: int) -> float:
            return score / (((5 + length) / 6) ** length_penalty)

        for step in range(1, self._max_new_tokens(max_new_tokens) + 1):
            logits = self.decode(memory.expand(beams.shape[0], -1, -1), beams)[:, -1].float()
            log_probs = self._forbid_special_tokens(logits).log_softmax(dim=-1)
            candidates = (scores[:, None] + log_probs).flatten()  # every beam x every token
            # 2 x beam_size candidates: enough to keep beam_size alive even if some end here.
            top_scores, top_ids = candidates.topk(min(2 * beam_size, candidates.numel()))
            rows, next_tokens = top_ids // cfg.vocab_size, top_ids % cfg.vocab_size

            alive: list[list[int]] = []
            alive_scores: list[float] = []
            for score, row, token in zip(top_scores.tolist(), rows.tolist(), next_tokens.tolist()):
                sequence = beams[row].tolist() + [token]
                if token == cfg.eos_token_id:
                    finished.append((sequence, normalised(score, step)))
                else:
                    alive.append(sequence)
                    alive_scores.append(score)
                if len(alive) == beam_size:
                    break
            if len(finished) >= beam_size or not alive:
                break
            beams = torch.tensor(alive, device=memory.device)
            scores = torch.tensor(alive_scores, device=memory.device)
        else:
            # Hit the length limit: unfinished beams are candidates too.
            length = beams.shape[1] - 1
            finished += [
                (b, normalised(s, length)) for b, s in zip(beams.tolist(), scores.tolist())
            ]

        finished.sort(key=lambda item: item[1], reverse=True)
        return [(torch.tensor(seq), score) for seq, score in finished[:beam_size]]

    # ------------------------------------------------------------- optimizer
    def parameter_groups(self, weight_decay: float, lr: float, encoder_lr: float) -> list[dict]:
        """AdamW parameter groups.

        - encoder params use `encoder_lr`, everything else uses `lr`;
        - weight decay only on Linear / Conv weight matrices; biases, norms,
          embeddings and position/class tokens get none;
        - frozen parameters are left out.
        named_parameters() lists the tied output/embedding weight only once.
        """
        decay_ids = {
            id(module.weight)
            for module in self.modules()
            if isinstance(module, (nn.Linear, nn.Conv2d))
        }
        decay_ids.discard(id(self.token_embedding.weight))  # tied with the output head

        groups: dict[tuple[bool, bool], list[nn.Parameter]] = {
            (is_enc, decay): [] for is_enc in (True, False) for decay in (True, False)
        }
        for name, param in self.named_parameters():
            if not param.requires_grad:
                continue
            groups[(name.startswith("encoder."), id(param) in decay_ids)].append(param)

        return [
            {
                "name": "encoder" if is_encoder else "decoder",  # shown in the training log
                "params": params,
                "lr": encoder_lr if is_encoder else lr,
                "weight_decay": weight_decay if decay else 0.0,
            }
            for (is_encoder, decay), params in groups.items()
            if params  # skip empty groups (e.g. frozen encoder)
        ]


def _sample(logits: Tensor, temperature: float, top_k: int | None, top_p: float | None) -> Tensor:
    """Pick one token per row from logits [B, V]."""
    if temperature <= 0.0:
        return logits.argmax(dim=-1)  # greedy
    logits = logits / temperature  # <1 sharper, >1 more random
    if top_k is not None:
        # Keep only the k best tokens; the rest get probability 0.
        kth_best = logits.topk(min(top_k, logits.shape[-1]), dim=-1).values[:, -1:]
        logits = logits.masked_fill(logits < kth_best, float("-inf"))
    if top_p is not None and top_p < 1.0:
        # Nucleus: keep the smallest set of best tokens whose probability sums to top_p.
        sorted_logits, order = logits.sort(dim=-1, descending=True)
        probs = sorted_logits.softmax(dim=-1)
        drop = probs.cumsum(dim=-1) - probs > top_p  # mass *before* this token already >= p
        sorted_logits = sorted_logits.masked_fill(drop, float("-inf"))
        logits = torch.full_like(logits, float("-inf")).scatter(-1, order, sorted_logits)
    probs = logits.softmax(dim=-1)
    return torch.multinomial(probs, num_samples=1).squeeze(1)


__all__ = [
    "IGNORE_INDEX",
    "BIAViT",
    "HuggingFaceEncoder",
    "ImageCaptioner",
    "TorchvisionResNet",
    "TorchvisionViT",
    "build_encoder",
    "encoder_normalization",
]

In [ ]:
%%writefile src/bia_captioner/metrics.py
"""Caption quality metrics: corpus BLEU-1..4 (Papineni et al., 2002).

Pure Python, no extra dependency. Matches `nltk.translate.bleu_score.corpus_bleu`
without smoothing: clipped n-gram precisions are summed over the whole corpus,
then combined with a brevity penalty that uses, for each hypothesis, the
reference length closest to it.
"""

import math
import re
from collections import Counter
from collections.abc import Sequence

_WORD = re.compile(r"[a-z0-9]+(?:'[a-z]+)?")


def tokenize(text: str) -> list[str]:
    """Lower-case words without punctuation: "A dog, running." -> ["a", "dog", "running"]."""
    return _WORD.findall(text.lower())


def _ngrams(words: Sequence[str], n: int) -> Counter:
    return Counter(tuple(words[i : i + n]) for i in range(len(words) - n + 1))


def corpus_bleu(
    hypotheses: Sequence[str], references: Sequence[Sequence[str]], max_n: int = 4
) -> dict[str, float]:
    """BLEU-1 .. BLEU-max_n of generated captions against their references.

    hypotheses: one generated caption per image.
    references: for each image, all of its human captions.
    Returns {"bleu1": ..., "bleu4": ...} with values in [0, 1].
    """
    if len(hypotheses) != len(references):
        raise ValueError("need exactly one list of references per hypothesis")
    matches = [0] * max_n
    totals = [0] * max_n
    hyp_len = ref_len = 0

    for hypothesis, refs in zip(hypotheses, references):
        hyp = tokenize(hypothesis)
        ref_tokens = [tokenize(r) for r in refs]
        hyp_len += len(hyp)
        # Closest reference length; ties go to the shorter one.
        ref_len += min((abs(len(r) - len(hyp)), len(r)) for r in ref_tokens)[1]
        for n in range(1, max_n + 1):
            counts = _ngrams(hyp, n)
            # Clip: an n-gram counts at most as often as in the reference that has it most.
            max_ref = Counter()
            for r in ref_tokens:
                max_ref |= _ngrams(r, n)
            matches[n - 1] += sum(min(c, max_ref[g]) for g, c in counts.items())
            totals[n - 1] += max(len(hyp) - n + 1, 0)

    if hyp_len == 0:
        return {f"bleu{n}": 0.0 for n in range(1, max_n + 1)}
    brevity = 1.0 if hyp_len > ref_len else math.exp(1 - ref_len / hyp_len)

    scores = {}
    log_sum = 0.0
    for n in range(1, max_n + 1):
        if matches[n - 1] == 0 or totals[n - 1] == 0:
            # One empty precision makes this and every higher-order BLEU zero.
            for m in range(n, max_n + 1):
                scores[f"bleu{m}"] = 0.0
            break
        log_sum += math.log(matches[n - 1] / totals[n - 1])
        scores[f"bleu{n}"] = brevity * math.exp(log_sum / n)
    return scores


def exact_match(hypotheses: Sequence[str], references: Sequence[Sequence[str]]) -> float:
    """Share of hypotheses equal to one of their references (same words, case and
    punctuation ignored). Meaningful when captions are short and unambiguous
    (the synthetic shapes data); for human captions BLEU is the usual metric."""
    if len(hypotheses) != len(references):
        raise ValueError("need exactly one list of references per hypothesis")
    if not hypotheses:
        return 0.0
    hits = sum(
        tokenize(h) in [tokenize(r) for r in refs] for h, refs in zip(hypotheses, references)
    )
    return hits / len(hypotheses)


__all__ = ["corpus_bleu", "exact_match", "tokenize"]

In [ ]:
%%writefile src/bia_captioner/train.py
"""Train the image captioning model.

The file is split into small pieces you can read top to bottom:

1. setup helpers   - seeding, device, mixed precision, multi-GPU (DDP) setup
2. data            - DataLoaders
3. optimisation    - AdamW + warmup/cosine learning-rate schedule
4. evaluation      - validation loss, perplexity, accuracy, BLEU, one decoded sample
5. checkpoints     - save / load everything needed to resume
6. train()         - the training loop (takes ready-made objects, easy to test)
7. CLI             - main() builds everything from command-line flags

Run it with:  python -m bia_captioner.train --help

Multi-GPU / multi-node: launch the same command with torchrun, e.g.
    torchrun --nproc_per_node=4 -m bia_captioner.train ...
torchrun starts one process per GPU and sets RANK / WORLD_SIZE / LOCAL_RANK;
the code below picks those up and wraps the model in DistributedDataParallel.
"""

from __future__ import annotations

import argparse
import dataclasses
import logging
import math
import os
import random
import time
from collections.abc import Callable, Iterator, Sequence
from contextlib import AbstractContextManager, contextmanager, nullcontext
from dataclasses import dataclass, field
from itertools import islice
from pathlib import Path
from typing import Any

import torch
import torch.distributed as dist
from torch import nn
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.optim import AdamW, Optimizer
from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import DataLoader, Dataset, DistributedSampler

from bia_captioner.config import (
    BIA_VIT,
    DATASET_PRESETS,
    TORCHVISION_RESNETS,
    TORCHVISION_VITS,
    DataConfig,
    ModelConfig,
    TrainConfig,
    config_to_dict,
    data_config_for,
    resolve_encoder_lr,
)
from bia_captioner.metrics import corpus_bleu
from bia_captioner.tokenizer import IGNORE_INDEX, vocab_size

try:  # numpy is optional; we only need it to seed its RNG
    import numpy as np
except ImportError:  # pragma: no cover
    np = None

logger = logging.getLogger(__name__)

LAST_CHECKPOINT = "last.pt"
BEST_CHECKPOINT = "best.pt"


# ---------------------------------------------------------------------------
# 1. Setup helpers
# ---------------------------------------------------------------------------


def seed_everything(seed: int) -> None:
    """Seed every random number generator we use."""
    random.seed(seed)
    if np is not None:
        np.random.seed(seed)
    torch.manual_seed(seed)  # also seeds all CUDA devices
    # We do NOT set torch.backends.cudnn.deterministic = True:
    # it makes convolutions slower, and tiny run-to-run noise is fine here.


def resolve_device(name: str = "auto") -> torch.device:
    """Turn "auto" | "cpu" | "cuda" | "mps" into a torch.device."""
    # Under torchrun each process owns one GPU: cuda:LOCAL_RANK.
    local_rank = int(os.environ.get("LOCAL_RANK", "0"))
    if name == "cuda" or (name == "auto" and torch.cuda.is_available()):
        return torch.device("cuda", local_rank)
    if name != "auto":
        return torch.device(name)
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def resolve_precision(precision: str, device: torch.device) -> tuple[torch.dtype | None, bool]:
    """Pick the autocast dtype and whether we need a GradScaler.

    Returns (dtype, use_scaler). dtype None means plain fp32 (no autocast).
    - bf16 has the same range as fp32, so it needs no loss scaling.
    - fp16 has a small range; small gradients underflow to 0 without a GradScaler.
    """
    if precision == "fp32":
        return None, False
    if precision == "bf16":
        return torch.bfloat16, False
    if precision == "fp16":
        if device.type != "cuda":
            raise ValueError("fp16 training needs a CUDA device")
        return torch.float16, True
    if precision == "auto":
        if device.type != "cuda":
            return None, False  # CPU / MPS: keep it simple and exact
        if torch.cuda.is_bf16_supported():
            return torch.bfloat16, False
        return torch.float16, True
    raise ValueError(f"unknown precision: {precision!r}")


@dataclass(frozen=True)
class DistInfo:
    """Who am I in a multi-process run? Defaults describe a single process."""

    rank: int = 0  # global id of this process (0 .. world_size-1)
    world_size: int = 1  # total number of processes (= total GPUs)
    local_rank: int = 0  # id of this process on its own machine

    @property
    def enabled(self) -> bool:
        return self.world_size > 1

    @property
    def is_main(self) -> bool:
        """Only the main process logs, decodes samples and writes checkpoints."""
        return self.rank == 0


def setup_distributed(device_name: str = "auto") -> DistInfo:
    """Join the process group if we were launched by torchrun; no-op otherwise."""
    world_size = int(os.environ.get("WORLD_SIZE", "1"))
    if world_size == 1:
        return DistInfo()
    local_rank = int(os.environ["LOCAL_RANK"])
    use_cuda = device_name in ("auto", "cuda") and torch.cuda.is_available()
    if use_cuda:
        torch.cuda.set_device(local_rank)  # this process only talks to its own GPU
    # NCCL is the fast backend for GPUs; gloo works on CPU (handy for testing).
    dist.init_process_group(backend="nccl" if use_cuda else "gloo")
    return DistInfo(dist.get_rank(), dist.get_world_size(), local_rank)


def get_dist_info() -> DistInfo:
    """Current process-group state (single process if none was started)."""
    if not (dist.is_available() and dist.is_initialized()):
        return DistInfo()
    local_rank = int(os.environ.get("LOCAL_RANK", "0"))
    return DistInfo(dist.get_rank(), dist.get_world_size(), local_rank)


@contextmanager
def local_main_first(info: DistInfo) -> Iterator[None]:
    """Let local rank 0 run the block first (downloads, cache building).

    The other processes on the same machine wait, then reuse the warm cache
    instead of all downloading the same files at once.
    """
    if info.enabled and info.local_rank != 0:
        dist.barrier()  # wait for the local main process
    yield
    if info.enabled and info.local_rank == 0:
        dist.barrier()  # release the others


def autocast(device: torch.device, dtype: torch.dtype | None) -> AbstractContextManager:
    """Mixed-precision context; does nothing when dtype is None."""
    return torch.autocast(device_type=device.type, dtype=dtype, enabled=dtype is not None)


def build_model_config(tokenizer: Any, data_config: DataConfig, **sizes: Any) -> ModelConfig:
    """ModelConfig whose vocab and special ids come straight from the tokenizer.

    `sizes` are the remaining ModelConfig fields (encoder, d_model, ...).
    """
    return ModelConfig(
        vocab_size=vocab_size(tokenizer),
        pad_token_id=tokenizer.pad_token_id,
        bos_token_id=tokenizer.bos_token_id,
        eos_token_id=tokenizer.eos_token_id,
        max_tokens=data_config.max_tokens,
        **sizes,
    )


# ---------------------------------------------------------------------------
# 2. Data
# ---------------------------------------------------------------------------


def seed_worker(worker_id: int) -> None:
    """Give each DataLoader worker its own numpy/python seed.

    PyTorch already seeds torch in each worker; we copy that seed to the others
    so random augmentations (and the random caption choice) differ between workers.
    """
    worker_seed = torch.initial_seed() % 2**32
    random.seed(worker_seed)
    if np is not None:
        np.random.seed(worker_seed)


def build_dataloaders(
    train_dataset: Dataset,
    val_dataset: Dataset,
    collate_fn: Any,
    config: TrainConfig,
    device: torch.device,
) -> tuple[DataLoader, DataLoader]:
    """Create the train and validation DataLoaders.

    With several processes, a DistributedSampler gives each one a different
    slice of the data, so batch_size is *per process* (per GPU).
    """
    info = get_dist_info()
    workers = config.num_workers
    common = {
        "batch_size": config.batch_size,
        "collate_fn": collate_fn,
        "num_workers": workers,
        "pin_memory": device.type == "cuda",  # faster CPU -> GPU copies
        "persistent_workers": workers > 0,  # do not restart workers every epoch
        "worker_init_fn": seed_worker if workers > 0 else None,
    }
    generator = torch.Generator()
    # Same order on every run; + rank so augmentation noise differs per process.
    generator.manual_seed(config.seed + info.rank)

    if info.enabled:
        train_sampler = DistributedSampler(
            train_dataset, shuffle=True, seed=config.seed, drop_last=True
        )
        # Note: to split evenly, the sampler may repeat a few val samples.
        val_sampler = DistributedSampler(val_dataset, shuffle=False, drop_last=False)
        train_loader = DataLoader(
            train_dataset, sampler=train_sampler, drop_last=True, generator=generator, **common
        )
        val_loader = DataLoader(val_dataset, sampler=val_sampler, drop_last=False, **common)
        return train_loader, val_loader

    train_loader = DataLoader(
        train_dataset,
        shuffle=True,
        drop_last=True,  # every step sees a full batch
        generator=generator,
        **common,
    )
    val_loader = DataLoader(val_dataset, shuffle=False, drop_last=False, **common)
    return train_loader, val_loader


def move_batch(batch: dict[str, torch.Tensor], device: torch.device) -> dict[str, torch.Tensor]:
    """Send every tensor of the batch to the device."""
    # non_blocking overlaps the copy with compute (works with pinned memory)
    return {k: v.to(device, non_blocking=True) for k, v in batch.items()}


# ---------------------------------------------------------------------------
# 3. Optimisation
# ---------------------------------------------------------------------------


def lr_lambda(step: int, warmup_steps: int, total_steps: int) -> float:
    """LR multiplier: linear warmup to 1, then cosine decay to 0.

    `step` is the number of optimizer steps already taken (starts at 0).
    """
    if step < warmup_steps:
        return (step + 1) / warmup_steps  # never exactly 0, so step 0 still learns
    decay_steps = max(1, total_steps - warmup_steps)
    progress = min(1.0, (step - warmup_steps) / decay_steps)
    return 0.5 * (1.0 + math.cos(math.pi * progress))


def build_optimizer(
    model: nn.Module, config: TrainConfig, device: torch.device, model_config: ModelConfig
) -> AdamW:
    """AdamW with the model's own parameter groups (no decay on norms/biases, etc.)."""
    groups = model.parameter_groups(
        weight_decay=config.weight_decay,
        lr=config.lr,
        encoder_lr=resolve_encoder_lr(config, model_config),
    )
    # fused=True runs the whole update in one CUDA kernel (faster); CUDA only.
    return AdamW(groups, lr=config.lr, betas=(0.9, 0.98), fused=device.type == "cuda")


def format_lrs(optimizer: Optimizer) -> str:
    """Current learning rate of each named param group, e.g. "decoder 3.0e-04 encoder 1.0e-05"."""
    lrs = {group.get("name", "lr"): group["lr"] for group in optimizer.param_groups}
    return " ".join(f"{name} {lr:.1e}" for name, lr in sorted(lrs.items()))


def build_scheduler(optimizer: Optimizer, warmup_steps: int, total_steps: int) -> LambdaLR:
    """Warmup + cosine schedule. Call scheduler.step() once per optimizer step."""
    return LambdaLR(optimizer, lambda step: lr_lambda(step, warmup_steps, total_steps))


# ---------------------------------------------------------------------------
# 4. Evaluation
# ---------------------------------------------------------------------------


def count_target_tokens(labels: torch.Tensor) -> torch.Tensor:
    """Number of tokens the loss is averaged over.

    The model predicts token t+1 from tokens <= t, so the first label is never
    a target. Padding (-100) is never a target either.
    """
    return (labels[:, 1:] != IGNORE_INDEX).sum()


def count_correct(logits: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
    """How many next tokens the model guessed right (top-1), ignoring padding.

    0.6 means 6 caption tokens out of 10 are predicted exactly, when the model
    sees the true caption so far.
    """
    predictions = logits[:, :-1].argmax(dim=-1)  # guess for token t+1 at position t
    targets = labels[:, 1:]
    return ((predictions == targets) & (targets != IGNORE_INDEX)).sum()


@torch.inference_mode()
def evaluate(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
    amp_dtype: torch.dtype | None = None,
) -> dict[str, float]:
    """Token-weighted mean loss, perplexity and token accuracy on a whole loader."""
    was_training = model.training
    model.eval()  # turns off dropout
    loss_sum = torch.zeros((), device=device, dtype=torch.float64)
    token_sum = torch.zeros((), device=device, dtype=torch.float64)
    correct_sum = torch.zeros((), device=device, dtype=torch.float64)
    for batch in loader:
        batch = move_batch(batch, device)
        with autocast(device, amp_dtype):
            logits, loss = model(batch["pixel_values"], batch["input_ids"], labels=batch["labels"])
        # Each batch has a different number of real tokens, so weight by it.
        n_tokens = count_target_tokens(batch["labels"])
        loss_sum += loss.double() * n_tokens
        token_sum += n_tokens
        correct_sum += count_correct(logits, batch["labels"])
    model.train(was_training)
    if dist.is_initialized():
        # Every process saw a different slice: add up the totals of all of them.
        for total in (loss_sum, token_sum, correct_sum):
            dist.all_reduce(total)
    tokens = token_sum.clamp(min=1)
    mean_loss = (loss_sum / tokens).item()  # one GPU sync at the end
    return {
        "loss": mean_loss,
        "perplexity": math.exp(min(mean_loss, 50.0)),
        "accuracy": (correct_sum / tokens).item(),
    }


@torch.inference_mode()
def caption_bleu(
    model: nn.Module,
    loader: DataLoader,
    tokenizer: Any,
    device: torch.device,
    amp_dtype: torch.dtype | None = None,
    max_images: int | None = None,
) -> dict[str, float] | None:
    """Greedy-caption up to `max_images` images and score them with corpus BLEU.

    Needs a dataset with `references(index)` and batches with an "index" key
    (bia_captioner.data datasets + CaptionCollator); returns None otherwise.
    """
    references_of = getattr(loader.dataset, "references", None)
    if references_of is None:
        return None
    was_training = model.training
    model.eval()
    hypotheses: list[str] = []
    references: list[list[str]] = []
    for batch in loader:
        if "index" not in batch or (max_images is not None and len(hypotheses) >= max_images):
            break
        with autocast(device, amp_dtype):
            tokens = model.generate(batch["pixel_values"].to(device), temperature=0.0)
        hypotheses += [tokenizer.decode(t, skip_special_tokens=True) for t in tokens.tolist()]
        references += [references_of(int(i)) for i in batch["index"]]
    model.train(was_training)
    if not hypotheses:
        return None
    n = len(hypotheses) if max_images is None else min(max_images, len(hypotheses))
    return corpus_bleu(hypotheses[:n], references[:n])


@torch.inference_mode()
def decode_sample(
    model: nn.Module,
    loader: DataLoader,
    tokenizer: Any,
    device: torch.device,
    amp_dtype: torch.dtype | None = None,
) -> tuple[str, str]:
    """Greedy-decode the first validation example. Returns (prediction, target)."""
    was_training = model.training
    model.eval()
    batch = next(iter(loader))
    pixel_values = batch["pixel_values"][:1].to(device)
    with autocast(device, amp_dtype):
        generated = model.generate(pixel_values, temperature=0.0)
    model.train(was_training)
    prediction = tokenizer.decode(generated[0].tolist(), skip_special_tokens=True)
    target = tokenizer.decode(batch["input_ids"][0].tolist(), skip_special_tokens=True)
    return prediction, target


def run_validation(
    model: nn.Module,
    val_loader: DataLoader,
    tokenizer: Any,
    device: torch.device,
    amp_dtype: torch.dtype | None,
    step: int,
    bleu_samples: int = 0,
) -> dict[str, Any]:
    """Evaluate, log the metrics, BLEU and one decoded sample, return all of them."""
    metrics: dict[str, Any] = evaluate(model, val_loader, device, amp_dtype)  # all processes
    if not get_dist_info().is_main:
        return metrics
    if bleu_samples > 0:
        bleu = caption_bleu(model, val_loader, tokenizer, device, amp_dtype, bleu_samples)
        if bleu is not None:
            metrics.update(bleu)
    bleu_text = f" | bleu4 {metrics['bleu4']:.4f}" if "bleu4" in metrics else ""
    logger.info(
        "val | step %d | loss %.4f | ppl %.2f | acc %.3f%s",
        step, metrics["loss"], metrics["perplexity"], metrics["accuracy"], bleu_text,
    )  # fmt: skip
    prediction, target = decode_sample(model, val_loader, tokenizer, device, amp_dtype)
    logger.info("sample target:     %s", target)
    logger.info("sample prediction: %s", prediction)
    metrics["sample_prediction"] = prediction
    metrics["sample_target"] = target
    return metrics


# ---------------------------------------------------------------------------
# 5. Checkpoints
# ---------------------------------------------------------------------------


@dataclass
class TrainState:
    """Counters that must survive a restart."""

    step: int = 0  # optimizer steps done
    epoch: int = 0  # epoch we are in
    best_val_loss: float = math.inf
    history: list[tuple[int, float]] = field(default_factory=list)  # (step, train loss)


def rng_state() -> dict[str, Any]:
    """Snapshot every RNG so a resumed run continues the same random stream."""
    state: dict[str, Any] = {"python": random.getstate(), "torch": torch.get_rng_state()}
    if np is not None:
        name, keys, pos, has_gauss, cached = np.random.get_state()
        # store plain lists so torch.load(weights_only=True) can read them back
        state["numpy"] = (name, keys.tolist(), pos, has_gauss, cached)
    if torch.cuda.is_available():
        state["cuda"] = torch.cuda.get_rng_state_all()
    return state


def set_rng_state(state: dict[str, Any]) -> None:
    """Restore what rng_state() saved."""
    random.setstate(state["python"])
    torch.set_rng_state(state["torch"])
    if np is not None and "numpy" in state:
        name, keys, pos, has_gauss, cached = state["numpy"]
        np.random.set_state((name, np.array(keys, dtype=np.uint32), pos, has_gauss, cached))
    if torch.cuda.is_available() and "cuda" in state:
        torch.cuda.set_rng_state_all(state["cuda"])


def save_checkpoint(
    path: Path,
    model: nn.Module,
    optimizer: Optimizer,
    scheduler: LambdaLR,
    scaler: torch.amp.GradScaler,
    state: TrainState,
    data_config: DataConfig,
    model_config: ModelConfig,
    train_config: TrainConfig,
) -> None:
    """Save everything needed to resume. Written atomically."""
    checkpoint = {
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "scaler": scaler.state_dict(),
        "step": state.step,
        "epoch": state.epoch,
        "best_val_loss": state.best_val_loss,
        "data_config": config_to_dict(data_config),
        "model_config": config_to_dict(model_config),
        "train_config": config_to_dict(train_config),
        "rng_state": rng_state(),
    }
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_suffix(path.suffix + ".tmp")
    torch.save(checkpoint, tmp_path)
    # os.replace is atomic: a crash mid-save never leaves a half-written file.
    os.replace(tmp_path, path)


def load_checkpoint(
    path: Path,
    model: nn.Module,
    optimizer: Optimizer | None = None,
    scheduler: LambdaLR | None = None,
    scaler: torch.amp.GradScaler | None = None,
) -> TrainState:
    """Load a checkpoint into the given objects and return the saved counters."""
    # weights_only=True refuses to run arbitrary pickled code (safe loading).
    checkpoint = torch.load(path, map_location="cpu", weights_only=True)
    model.load_state_dict(checkpoint["model"])
    if optimizer is not None:
        optimizer.load_state_dict(checkpoint["optimizer"])
    if scheduler is not None:
        scheduler.load_state_dict(checkpoint["scheduler"])
    if scaler is not None and checkpoint["scaler"]:  # empty dict when scaler disabled
        scaler.load_state_dict(checkpoint["scaler"])
    set_rng_state(checkpoint["rng_state"])
    return TrainState(
        step=checkpoint["step"],
        epoch=checkpoint["epoch"],
        best_val_loss=checkpoint["best_val_loss"],
    )


# ---------------------------------------------------------------------------
# 6. Training loop
# ---------------------------------------------------------------------------


def epoch_batches(loader: DataLoader, epoch: int, seed: int, skip: int) -> Iterator[dict]:
    """Batches of one epoch, in a shuffle order that depends only on (seed, epoch).

    Re-seeding per epoch means a resumed run sees the same order, so we can
    skip the batches that were already used before the restart.
    """
    if isinstance(loader.sampler, DistributedSampler):
        loader.sampler.set_epoch(epoch)  # new shuffle each epoch, same on every process
    elif loader.generator is not None:
        loader.generator.manual_seed(seed + epoch)
    return islice(iter(loader), skip, None)


def train(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    tokenizer: Any,
    data_config: DataConfig,
    model_config: ModelConfig,
    train_config: TrainConfig,
    callback: Callable[[dict[str, Any]], None] | None = None,
) -> TrainState:
    """Run the full training loop and return the final counters.

    Everything is passed in already built, so tests can use tiny fakes.
    `callback` (optional) receives every logged metric as a dict, on the main
    process only; the notebook uses it to draw live training curves:
        {"kind": "train", "step", "epoch", "total_steps", "loss", "accuracy",
         "lr": {group: lr}, "grad_norm", "tokens_per_sec"}
        {"kind": "val", "step", "epoch", "loss", "perplexity", "accuracy",
         "bleu1".."bleu4" (when computed), "sample_prediction", "sample_target"}
    """
    cfg = train_config
    info = get_dist_info()
    device = resolve_device(cfg.device)
    amp_dtype, use_scaler = resolve_precision(cfg.precision, device)
    model.to(device)
    if cfg.freeze_encoder:
        model.freeze_encoder(True)  # before building the optimizer: frozen params are left out
    model.train()
    raw_model = model  # the plain model: used for checkpoints, eval and generation

    # One optimizer step = grad_accum_steps micro-batches.
    accum = cfg.grad_accum_steps
    steps_per_epoch = len(train_loader) // accum
    if steps_per_epoch == 0:
        raise ValueError("train set too small for batch_size * grad_accum_steps")
    total_steps = cfg.epochs * steps_per_epoch

    optimizer = build_optimizer(raw_model, cfg, device, model_config)
    scheduler = build_scheduler(optimizer, cfg.warmup_steps, total_steps)
    scaler = torch.amp.GradScaler("cuda", enabled=use_scaler)

    output_dir = Path(cfg.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    if info.is_main:
        tokenizer.save_pretrained(output_dir / "tokenizer")  # inference needs the same vocab

    state = TrainState()
    if cfg.resume is not None:
        state = load_checkpoint(Path(cfg.resume), raw_model, optimizer, scheduler, scaler)
        logger.info("resumed from %s at step %d", cfg.resume, state.step)

    if info.enabled:
        # DDP averages the gradients of all processes after each backward pass.
        # It also copies rank 0's weights to everyone at start, so all begin equal.
        model = DDP(model, device_ids=[device.index] if device.type == "cuda" else None)

    def checkpoint(path: Path) -> None:
        if not info.is_main:
            return  # one writer is enough: all processes hold the same weights
        save_checkpoint(
            path, raw_model, optimizer, scheduler, scaler, state,
            data_config, model_config, cfg,
        )  # fmt: skip

    def emit(event: dict[str, Any]) -> None:
        if callback is not None and info.is_main:
            callback(event)

    def validate_and_save() -> None:
        metrics = run_validation(
            raw_model, val_loader, tokenizer, device, amp_dtype, state.step, cfg.bleu_samples
        )
        emit({"kind": "val", "step": state.step, "epoch": state.epoch, **metrics})
        val_loss = metrics["loss"]
        if val_loss < state.best_val_loss:
            state.best_val_loss = val_loss
            checkpoint(output_dir / BEST_CHECKPOINT)
        checkpoint(output_dir / LAST_CHECKPOINT)

    logger.info(
        "training: %d epochs x %d steps (effective batch %d = %d per process x %d accum x %d "
        "processes) on %s, amp=%s, encoder=%s%s",
        cfg.epochs, steps_per_epoch, cfg.batch_size * accum * info.world_size,
        cfg.batch_size, accum, info.world_size, device, amp_dtype,
        model_config.encoder, " (frozen)" if cfg.freeze_encoder else "",
    )  # fmt: skip

    # Running sums for the log line (kept on device: no GPU sync every step).
    window_loss = torch.zeros((), device=device)
    window_tokens = torch.zeros((), device=device)  # real tokens, for tok/s
    window_targets = torch.zeros((), device=device)  # predicted tokens, for accuracy
    window_correct = torch.zeros((), device=device)
    window_images = 0
    window_start = time.perf_counter()

    for epoch in range(state.step // steps_per_epoch, cfg.epochs):
        state.epoch = epoch
        done_in_epoch = state.step - epoch * steps_per_epoch  # > 0 only right after resume
        batches = epoch_batches(train_loader, epoch, cfg.seed, skip=done_in_epoch * accum)
        micro_batches = (steps_per_epoch - done_in_epoch) * accum  # drop the leftover tail

        for micro_step, batch in enumerate(islice(batches, micro_batches)):
            batch = move_batch(batch, device)
            is_update_step = (micro_step + 1) % accum == 0
            # With DDP, skip the gradient all-reduce until the last micro-batch:
            # syncing once per optimizer step is enough and much cheaper.
            sync = model.no_sync() if info.enabled and not is_update_step else nullcontext()
            with sync:
                with autocast(device, amp_dtype):
                    logits, loss = model(
                        batch["pixel_values"],
                        batch["input_ids"],
                        labels=batch["labels"],
                        label_smoothing=cfg.label_smoothing,
                    )
                # Divide so the summed gradients equal the gradient of the mean loss.
                scaler.scale(loss / accum).backward()
            window_loss += loss.detach() / accum
            window_tokens += (batch["labels"] != IGNORE_INDEX).sum()
            window_targets += count_target_tokens(batch["labels"])
            window_correct += count_correct(logits.detach(), batch["labels"])
            window_images += batch["pixel_values"].shape[0]

            if not is_update_step:
                continue  # keep accumulating gradients

            # --- one optimizer step ---
            scaler.unscale_(optimizer)  # back to real gradient values before clipping
            max_norm = cfg.grad_clip if cfg.grad_clip > 0 else math.inf
            grad_norm = nn.utils.clip_grad_norm_(model.parameters(), max_norm)
            scaler.step(optimizer)  # skips the step if fp16 gradients overflowed
            scaler.update()
            optimizer.zero_grad(set_to_none=True)  # frees memory, slightly faster
            scheduler.step()
            state.step += 1

            if state.step % cfg.log_every == 0:
                elapsed = time.perf_counter() - window_start
                mean_loss = window_loss.item() / cfg.log_every
                accuracy = (window_correct / window_targets.clamp(min=1)).item()
                # loss/accuracy are this process's; speeds are estimated for all processes
                tokens_per_sec = window_tokens.item() * info.world_size / elapsed
                images_per_sec = window_images * info.world_size / elapsed
                state.history.append((state.step, mean_loss))
                logger.info(
                    "train | epoch %d | step %d/%d | loss %.4f | acc %.3f | lr %s | grad %.2f "
                    "| %.0f img/s",
                    epoch, state.step, total_steps, mean_loss, accuracy, format_lrs(optimizer),
                    grad_norm.item(), images_per_sec,
                )  # fmt: skip
                emit({
                    "kind": "train", "step": state.step, "epoch": epoch,
                    "total_steps": total_steps, "loss": mean_loss, "accuracy": accuracy,
                    "lr": {g.get("name", "lr"): g["lr"] for g in optimizer.param_groups},
                    "grad_norm": grad_norm.item(), "tokens_per_sec": tokens_per_sec,
                    "images_per_sec": images_per_sec,
                })  # fmt: skip
                for total in (window_loss, window_tokens, window_targets, window_correct):
                    total.zero_()
                window_images = 0
                window_start = time.perf_counter()

            # Validate every eval_every steps, and once more at the very end.
            if state.step % cfg.eval_every == 0 or state.step == total_steps:
                validate_and_save()
                window_start = time.perf_counter()  # don't count eval time in the speed

    state.epoch = cfg.epochs
    checkpoint(output_dir / LAST_CHECKPOINT)
    logger.info("done | best val loss %.4f", state.best_val_loss)
    return state


# ---------------------------------------------------------------------------
# 7. Command line
# ---------------------------------------------------------------------------


ENCODER_HELP = (
    f"image encoder: {BIA_VIT!r} (own ViT, from scratch), a torchvision ViT "
    f"{TORCHVISION_VITS}, a torchvision ResNet {TORCHVISION_RESNETS}, or any Hugging Face "
    "vision model as 'hf:<model id>' (e.g. hf:google/vit-base-patch16-224-in21k)"
)


def build_parser() -> argparse.ArgumentParser:
    """All CLI flags. Defaults come from the config dataclasses."""
    d, t = DataConfig(), TrainConfig()
    m = ModelConfig(vocab_size=1, pad_token_id=0, bos_token_id=1, eos_token_id=1)
    p = argparse.ArgumentParser(description="Train the BIA image captioning model.")

    data = p.add_argument_group("data")
    data.add_argument(
        "--dataset", choices=list(DATASET_PRESETS), default="flickr30k",
        help="dataset preset (see docs/datasets.md); the flags below override it",
    )  # fmt: skip
    data.add_argument("--dataset-id", default=None, help="Hugging Face dataset id")
    data.add_argument("--image-column", default=None)
    data.add_argument("--caption-columns", nargs="+", default=None)
    data.add_argument("--caption-separator", default=None, help="split caption cells on it")
    data.add_argument("--group-column", default=None, help="rows sharing it are one image")
    data.add_argument(
        "--holdout", type=float, nargs=2, metavar=("VAL", "TEST"), default=None,
        help="single-split dataset: fractions held out as validation / test",
    )  # fmt: skip
    data.add_argument("--tokenizer-id", default=d.tokenizer_id)
    data.add_argument("--cache-dir", type=Path, default=d.cache_dir)
    data.add_argument("--train-split", default=d.train_split)
    data.add_argument("--val-split", default=d.val_split)
    data.add_argument("--image-size", type=int, default=d.image_size)
    data.add_argument("--max-tokens", type=int, default=d.max_tokens)

    model = p.add_argument_group("model")
    model.add_argument("--encoder", default=m.encoder, help=ENCODER_HELP)
    model.add_argument(
        "--from-scratch", action="store_true",
        help="random-init the encoder instead of loading pretrained weights",
    )  # fmt: skip
    model.add_argument("--d-model", type=int, default=m.d_model)
    model.add_argument("--heads", type=int, default=m.attention_heads)
    model.add_argument("--layers", type=int, default=m.decoder_layers, help="decoder layers")
    model.add_argument("--ffn-mult", type=int, default=m.ffn_mult)
    model.add_argument("--dropout", type=float, default=m.dropout)
    model.add_argument("--patch-size", type=int, default=m.patch_size, help="bia-vit only")
    model.add_argument("--encoder-layers", type=int, default=m.encoder_layers, help="bia-vit only")

    opt = p.add_argument_group("training")
    opt.add_argument("--epochs", type=int, default=t.epochs)
    opt.add_argument("--batch-size", type=int, default=t.batch_size)
    opt.add_argument("--grad-accum", type=int, default=t.grad_accum_steps)
    opt.add_argument("--lr", type=float, default=t.lr)
    opt.add_argument(
        "--encoder-lr", type=float, default=t.encoder_lr,
        help="default: 1e-5 for a pretrained encoder, = --lr for one trained from scratch",
    )  # fmt: skip
    opt.add_argument("--freeze-encoder", action="store_true", help="train only the decoder")
    opt.add_argument("--weight-decay", type=float, default=t.weight_decay)
    opt.add_argument("--warmup-steps", type=int, default=t.warmup_steps)
    opt.add_argument("--grad-clip", type=float, default=t.grad_clip)
    opt.add_argument("--label-smoothing", type=float, default=t.label_smoothing)
    opt.add_argument("--no-augment", action="store_true", help="no random crop / colour jitter")
    opt.add_argument("--precision", choices=["auto", "fp32", "bf16", "fp16"], default=t.precision)
    opt.add_argument("--num-workers", type=int, default=t.num_workers)
    opt.add_argument("--train-limit", type=int, default=t.train_limit)
    opt.add_argument("--val-limit", type=int, default=t.val_limit)
    opt.add_argument("--log-every", type=int, default=t.log_every)
    opt.add_argument("--eval-every", type=int, default=t.eval_every)
    opt.add_argument("--bleu-samples", type=int, default=t.bleu_samples)
    opt.add_argument("--device", default=t.device)
    opt.add_argument("--output-dir", type=Path, default=t.output_dir)
    opt.add_argument("--resume", type=Path, default=t.resume)
    opt.add_argument("--seed", type=int, default=t.seed)
    opt.add_argument(
        "--overfit",
        type=int,
        metavar="N",
        default=None,
        help="sanity check: train and validate on the same N train images, no regularisation",
    )
    return p


def configs_from_args(args: argparse.Namespace) -> tuple[DataConfig, TrainConfig]:
    """Turn parsed flags into DataConfig and TrainConfig."""
    overrides = {
        "dataset_id": args.dataset_id,
        "image_column": args.image_column,
        "caption_columns": tuple(args.caption_columns) if args.caption_columns else None,
        "caption_separator": args.caption_separator,
        "group_column": args.group_column,
        "holdout": tuple(args.holdout) if args.holdout else None,
    }
    data_config = data_config_for(
        args.dataset,
        **{k: v for k, v in overrides.items() if v is not None},
        tokenizer_id=args.tokenizer_id,
        cache_dir=args.cache_dir,
        train_split=args.train_split,
        val_split=args.val_split,
        image_size=args.image_size,
        max_tokens=args.max_tokens,
    )
    train_config = TrainConfig(
        output_dir=args.output_dir,
        seed=args.seed,
        epochs=args.epochs,
        batch_size=args.batch_size,
        grad_accum_steps=args.grad_accum,
        lr=args.lr,
        encoder_lr=args.encoder_lr,
        freeze_encoder=args.freeze_encoder,
        weight_decay=args.weight_decay,
        warmup_steps=args.warmup_steps,
        grad_clip=args.grad_clip,
        label_smoothing=args.label_smoothing,
        augment=not args.no_augment,
        precision=args.precision,
        num_workers=args.num_workers,
        train_limit=args.train_limit,
        val_limit=args.val_limit,
        log_every=args.log_every,
        eval_every=args.eval_every,
        bleu_samples=args.bleu_samples,
        device=args.device,
        resume=args.resume,
    )
    if args.overfit is not None:
        # "Can the model memorise a tiny set?" If not, something is broken.
        n = args.overfit
        data_config = dataclasses.replace(data_config, val_split=data_config.train_split)
        train_config = dataclasses.replace(
            train_config,
            train_limit=n,
            val_limit=n,
            batch_size=min(train_config.batch_size, n),  # drop_last would leave 0 batches
            label_smoothing=0.0,  # smoothing keeps the loss above 0 even when memorised
            augment=False,  # memorise exactly these images
            bleu_samples=min(train_config.bleu_samples, n),
        )
        args.dropout = 0.0
    return data_config, train_config


def model_sizes_from_args(args: argparse.Namespace) -> dict[str, Any]:
    """The ModelConfig fields that come from the CLI."""
    return {
        "encoder": args.encoder,
        "pretrained": not args.from_scratch,
        "d_model": args.d_model,
        "attention_heads": args.heads,
        "decoder_layers": args.layers,
        "ffn_mult": args.ffn_mult,
        "dropout": args.dropout,
        "patch_size": args.patch_size,
        "encoder_layers": args.encoder_layers,
    }


def main(argv: Sequence[str] | None = None) -> None:
    """CLI entry point: set up (optional) multi-process training, then train.

    Returns None on purpose: console scripts pass the return value to sys.exit,
    and anything else would be printed and turned into exit code 1.
    """
    args = build_parser().parse_args(argv)
    data_config, train_config = configs_from_args(args)
    info = setup_distributed(train_config.device)  # no-op unless launched by torchrun

    # Only the main process prints INFO lines; the others only warnings.
    logging.basicConfig(
        level=logging.INFO if info.is_main else logging.WARNING,
        format="%(asctime)s %(levelname)s %(message)s",
        datefmt="%H:%M:%S",
    )
    for name in ("httpx", "httpx2"):  # hide Hugging Face HTTP logs
        logging.getLogger(name).setLevel(logging.WARNING)
    try:
        _build_and_train(args, data_config, train_config, info)
    finally:
        if info.enabled:
            dist.destroy_process_group()


def _build_and_train(
    args: argparse.Namespace, data_config: DataConfig, train_config: TrainConfig, info: DistInfo
) -> TrainState:
    """Build tokenizer, data and model from the CLI flags, then train."""
    # Heavy imports here so `import bia_captioner.train` stays light for tests.
    from bia_captioner.data import CaptionCollator, build_dataset
    from bia_captioner.model import ImageCaptioner, encoder_normalization
    from bia_captioner.tokenizer import build_tokenizer

    seed_everything(train_config.seed)  # before building the model: same init weights
    with local_main_first(info):  # one download per machine, the rest reuse the cache
        tokenizer = build_tokenizer(data_config.tokenizer_id, data_config.cache_dir)
        model_config = build_model_config(tokenizer, data_config, **model_sizes_from_args(args))
        # Images must be normalised the way the encoder was trained (saved in checkpoints).
        mean, std = encoder_normalization(model_config)
        data_config = dataclasses.replace(data_config, image_mean=mean, image_std=std)
        train_dataset = build_dataset(
            data_config, tokenizer, data_config.train_split,
            limit=train_config.train_limit, train=train_config.augment,
        )  # fmt: skip
        val_dataset = build_dataset(
            data_config, tokenizer, data_config.val_split, limit=train_config.val_limit
        )
        model = ImageCaptioner(model_config, image_size=data_config.image_size)

    collator = CaptionCollator(pad_token_id=tokenizer.pad_token_id)
    device = resolve_device(train_config.device)
    train_loader, val_loader = build_dataloaders(
        train_dataset, val_dataset, collator, train_config, device
    )

    n_params = sum(p.numel() for p in model.parameters())
    n_encoder = sum(p.numel() for p in model.encoder.parameters())
    logger.info(
        "model: %.1fM parameters (encoder %s, %s, %.1fM) | data %s | train %d images | "
        "val %d images",
        n_params / 1e6, model_config.encoder,
        "pretrained" if model_config.encoder_is_pretrained else "from scratch",
        n_encoder / 1e6, data_config.dataset_id, len(train_dataset), len(val_dataset),
    )  # fmt: skip

    return train(
        model, train_loader, val_loader, tokenizer, data_config, model_config, train_config
    )


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/bia_captioner/generate.py
"""Use a trained model: image in, caption out.

    python -m bia_captioner.generate --checkpoint checkpoints/run/best.pt --image photo.jpg

Everything needed to rebuild the network (its configs) is stored inside the
checkpoint, and the tokenizer is saved next to it, so no flags have to match.
"""

from __future__ import annotations

import argparse
import json
import logging
from collections.abc import Sequence
from pathlib import Path

import torch
from PIL import Image
from transformers import AutoTokenizer, PreTrainedTokenizerBase

from bia_captioner.config import DataConfig, ModelConfig
from bia_captioner.data import build_image_transform
from bia_captioner.model import ImageCaptioner
from bia_captioner.train import resolve_device

logger = logging.getLogger(__name__)


def load_model(
    checkpoint_path: Path, device: torch.device | str = "cpu"
) -> tuple[ImageCaptioner, DataConfig]:
    """Rebuild the model from a checkpoint and put it in eval mode."""
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    model_config = ModelConfig(**checkpoint["model_config"])
    data_values = checkpoint["data_config"]
    data_config = DataConfig(**{**data_values, "cache_dir": None})  # cache path is machine-specific

    # pretrained_encoder=False: the trained weights come from the checkpoint anyway.
    model = ImageCaptioner(model_config, data_config.image_size, pretrained_encoder=False)
    model.load_state_dict(checkpoint["model"])
    model.to(device).eval()  # eval: no dropout, BatchNorm uses its running stats
    return model, data_config


def load_tokenizer(checkpoint_path: Path) -> PreTrainedTokenizerBase:
    """The tokenizer that train.py saved next to the checkpoint."""
    return AutoTokenizer.from_pretrained(Path(checkpoint_path).parent / "tokenizer")


@torch.inference_mode()
def caption_image(
    model: ImageCaptioner,
    tokenizer: PreTrainedTokenizerBase,
    image: Image.Image,
    data_config: DataConfig,
    beam_size: int = 3,
    length_penalty: float = 0.6,
    temperature: float = 0.0,
    top_k: int | None = None,
    top_p: float | None = None,
) -> str:
    """Caption one image.

    beam_size > 1 uses beam search (best quality, deterministic). With
    beam_size 1, temperature 0 is greedy and temperature > 0 samples
    (optionally restricted with top_k / top_p).
    """
    device = next(model.parameters()).device
    transform = build_image_transform(
        data_config.image_size, False, data_config.image_mean, data_config.image_std
    )
    pixel_values = transform(image.convert("RGB"))
    pixel_values = pixel_values[None].to(device)
    if beam_size > 1:
        tokens, _ = model.beam_search(
            pixel_values, beam_size=beam_size, length_penalty=length_penalty
        )[0]
    else:
        tokens = model.generate(pixel_values, temperature=temperature, top_k=top_k, top_p=top_p)[0]
    return tokenizer.decode(tokens.tolist(), skip_special_tokens=True).strip()


def build_parser() -> argparse.ArgumentParser:
    p = argparse.ArgumentParser(description="Caption images with a trained BIA captioner.")
    p.add_argument("--checkpoint", type=Path, required=True, help="path to best.pt / last.pt")
    p.add_argument("--image", type=Path, required=True, nargs="+", help="one or more images")
    p.add_argument("--output", type=Path, default=None, help="save {image: caption} as JSON")
    p.add_argument("--beam-size", type=int, default=3, help="1 = greedy / sampling")
    p.add_argument("--length-penalty", type=float, default=0.6, help="beam search only")
    p.add_argument("--temperature", type=float, default=0.0, help="beam-size 1 only; 0 = greedy")
    p.add_argument("--top-k", type=int, default=None, help="sample only from the k best tokens")
    p.add_argument("--top-p", type=float, default=None, help="nucleus sampling threshold")
    p.add_argument("--device", default="auto")
    return p


def main(argv: Sequence[str] | None = None) -> None:
    """CLI entry point (returns None: console scripts pass the return value to sys.exit)."""
    run(argv)


def run(argv: Sequence[str] | None = None) -> dict[str, str]:
    """Parse the flags, caption every image and return {image path: caption}."""
    logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
    args = build_parser().parse_args(argv)
    device = resolve_device(args.device)
    model, data_config = load_model(args.checkpoint, device)
    tokenizer = load_tokenizer(args.checkpoint)

    captions = {}
    for image_path in args.image:
        caption = caption_image(
            model, tokenizer, Image.open(image_path), data_config,
            beam_size=args.beam_size, length_penalty=args.length_penalty,
            temperature=args.temperature, top_k=args.top_k, top_p=args.top_p,
        )  # fmt: skip
        captions[str(image_path)] = caption
        print(f"{image_path}: {caption}")
    if args.output is not None:
        args.output.parent.mkdir(parents=True, exist_ok=True)
        args.output.write_text(json.dumps(captions, indent=2) + "\n")
        logger.info("wrote %s", args.output)
    return captions


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/bia_captioner/evaluate.py
"""Score a trained model on a held-out split with corpus BLEU-1..4.

    python -m bia_captioner.evaluate --checkpoint checkpoints/run/best.pt --split test

Each image is captioned once (beam search by default) and compared with all
of its human captions. Optionally writes every prediction to a JSON file.
"""

from __future__ import annotations

import argparse
import dataclasses
import json
import logging
import textwrap
from collections.abc import Sequence
from pathlib import Path

import torch
from PIL import Image, ImageDraw, ImageFont
from torch.utils.data import DataLoader, Subset

from bia_captioner.data import CaptionCollator, build_dataset
from bia_captioner.generate import load_model, load_tokenizer
from bia_captioner.metrics import corpus_bleu, exact_match
from bia_captioner.train import autocast, resolve_device, resolve_precision

logger = logging.getLogger(__name__)


@torch.inference_mode()
def predict(
    model: torch.nn.Module,
    dataset: torch.utils.data.Dataset,
    tokenizer,
    device: torch.device,
    beam_size: int = 3,
    length_penalty: float = 0.6,
    batch_size: int = 32,
    num_workers: int = 2,
    amp_dtype: torch.dtype | None = None,
    shuffle_images: bool = False,
) -> list[str]:
    """Caption every image of the dataset, in order.

    shuffle_images=True is a sanity check: caption i is generated from another,
    randomly chosen image of the split (a fixed random permutation), so scores
    must collapse if the model really reads the image and stay the same if it
    only learned the language.
    """
    if shuffle_images:
        dataset = Subset(dataset, shuffled_order(len(dataset)))
    loader = DataLoader(
        dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers,
        collate_fn=CaptionCollator(pad_token_id=tokenizer.pad_token_id),
    )  # fmt: skip
    captions: list[str] = []
    for batch in loader:
        pixel_values = batch["pixel_values"].to(device)
        with autocast(device, amp_dtype):
            if beam_size > 1:  # beam search works on one image at a time
                sequences = [
                    model.beam_search(image[None], beam_size, length_penalty=length_penalty)[0][0]
                    for image in pixel_values
                ]
            else:  # greedy, the whole batch at once
                sequences = list(model.generate(pixel_values))
        captions += [
            tokenizer.decode(s.tolist(), skip_special_tokens=True).strip() for s in sequences
        ]
        logger.info("captioned %d / %d images", len(captions), len(dataset))
    return captions


def shuffled_order(n: int, seed: int = 0) -> list[int]:
    """For each image i, another image to take its pixels from (fixed, random).

    The images are put in a random cycle and each one takes the next one's
    place, so no image is ever paired with itself.
    """
    cycle = torch.randperm(n, generator=torch.Generator().manual_seed(seed)).tolist()
    source = list(range(n))
    for k, image in enumerate(cycle):
        source[image] = cycle[(k + 1) % n]
    return source


def spread_indices(n: int, count: int) -> list[int]:
    """`count` indices spread evenly over range(n) (datasets are often sorted by class)."""
    count = min(count, n)
    return sorted({round(i * (n - 1) / max(count - 1, 1)) for i in range(count)})


def save_gallery(
    images: Sequence[Image.Image],
    predictions: Sequence[str],
    references: Sequence[Sequence[str]],
    path: Path,
    columns: int = 4,
    tile: int = 320,
) -> None:
    """Grid of images with the model's caption (and one human caption) under each.

    Made to be shared: a README, a slide or a post.
    """
    font = ImageFont.load_default(size=17)
    small = ImageFont.load_default(size=14)
    chars = int(tile / 8.6)  # characters per line at these font sizes
    text_height = 84
    rows = -(-len(images) // columns)
    sheet = Image.new("RGB", (columns * tile, rows * (tile + text_height)), "white")
    draw = ImageDraw.Draw(sheet)
    for i, (image, prediction, refs) in enumerate(zip(images, predictions, references)):
        x, y = (i % columns) * tile, (i // columns) * (tile + text_height)
        side = min(image.size)  # centre square crop, like a thumbnail
        left, top = (image.width - side) // 2, (image.height - side) // 2
        square = image.convert("RGB").crop((left, top, left + side, top + side))
        sheet.paste(square.resize((tile - 8, tile - 8), Image.LANCZOS), (x + 4, y + 4))
        lines = textwrap.wrap(prediction, chars)[:3]
        draw.multiline_text((x + 8, y + tile), "\n".join(lines), fill="black", font=font)
        human = textwrap.shorten("human: " + refs[0], width=int(chars * 1.2), placeholder="...")
        draw.text((x + 8, y + tile + 18 * len(lines) + 6), human, fill=(110, 110, 110), font=small)
    path.parent.mkdir(parents=True, exist_ok=True)
    sheet.save(path)


def build_parser() -> argparse.ArgumentParser:
    p = argparse.ArgumentParser(description="BLEU evaluation of a trained BIA captioner.")
    p.add_argument("--checkpoint", type=Path, required=True)
    p.add_argument("--split", default="test", help="dataset split to score (default: test)")
    p.add_argument("--limit", type=int, default=None, help="score only the first N images")
    p.add_argument("--beam-size", type=int, default=3, help="1 = greedy")
    p.add_argument("--length-penalty", type=float, default=0.6)
    p.add_argument("--batch-size", type=int, default=32)
    p.add_argument("--num-workers", type=int, default=2)
    p.add_argument("--cache-dir", type=Path, default=None)
    p.add_argument("--output", type=Path, default=None, help="write scores + predictions (JSON)")
    p.add_argument("--gallery", type=Path, default=None, help="save a captioned image grid (PNG)")
    p.add_argument("--gallery-size", type=int, default=12, help="images in the gallery")
    p.add_argument(
        "--shuffle-images", action="store_true",
        help="sanity check: caption each image from another image's pixels (scores must drop)",
    )  # fmt: skip
    p.add_argument("--device", default="auto")
    return p


def main(argv: Sequence[str] | None = None) -> None:
    """CLI entry point (returns None: console scripts pass the return value to sys.exit)."""
    run(argv)


def run(argv: Sequence[str] | None = None, dataset_loader=None) -> dict[str, float]:
    """Caption the split, print BLEU and return the scores."""
    logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
    for name in ("httpx", "httpx2"):  # hide Hugging Face HTTP logs
        logging.getLogger(name).setLevel(logging.WARNING)
    args = build_parser().parse_args(argv)
    device = resolve_device(args.device)
    amp_dtype, _ = resolve_precision("auto", device)
    model, data_config = load_model(args.checkpoint, device)
    data_config = dataclasses.replace(data_config, cache_dir=args.cache_dir)
    tokenizer = load_tokenizer(args.checkpoint)

    dataset = build_dataset(
        data_config, tokenizer, args.split, limit=args.limit, dataset_loader=dataset_loader
    )
    predictions = predict(
        model, dataset, tokenizer, device, args.beam_size, args.length_penalty,
        args.batch_size, args.num_workers, amp_dtype, args.shuffle_images,
    )  # fmt: skip
    references = [dataset.references(i) for i in range(len(dataset))]
    scores = corpus_bleu(predictions, references)
    scores["exact_match"] = exact_match(predictions, references)

    shuffled = " | IMAGES SHUFFLED (sanity check)" if args.shuffle_images else ""
    print(f"{args.split}: {len(dataset)} images | beam {args.beam_size}{shuffled}")
    for name, value in scores.items():
        print(f"  {name.upper():11} {100 * value:6.2f}")
    if args.output is not None:
        args.output.parent.mkdir(parents=True, exist_ok=True)
        rows = [{"prediction": p, "references": r} for p, r in zip(predictions, references)]
        args.output.write_text(json.dumps({"scores": scores, "predictions": rows}, indent=2))
        logger.info("wrote %s", args.output)
    if args.gallery is not None:
        picks = spread_indices(len(dataset), args.gallery_size)
        save_gallery(
            [dataset.image(i) for i in picks],
            [predictions[i] for i in picks],
            [references[i] for i in picks],
            args.gallery,
        )
        logger.info("wrote %s", args.gallery)
    return scores


if __name__ == "__main__":
    main()

In [ ]:
import importlib
import sys

sys.path.insert(0, "src")
for name in [m for m in sys.modules if m.startswith("bia_captioner")]:
    del sys.modules[name]  # pick up the files just written if this cell is re-run
import bia_captioner.train as T

importlib.reload(T)
print("bia_captioner ready")

## 3. Settings

**Dataset** (`DATASET`, see `docs/datasets.md`):

| `DATASET` | Images | What for |
|---|---|---|
| `"shapes"` | 8 000 generated shapes, no download | prove the architecture learns from pixels (minutes) |
| `"flowers"` | 6 552 flower photos, 1 simple caption each, ~0.3 GB | the "CIFAR-10 of captioning": good results fast, nice to show |
| `"birds"` | 5 994 bird photos, ~10 human captions each, ~0.6 GB | fine-grained human descriptions |
| `"flickr8k"` | 8 091 photos, ~0.3 GB | first real-photo run, fast |
| `"flickr30k"` | 31 014 photos, ~4.3 GB | the default real benchmark |
| `"coco"` | 123 287 photos, ~21 GB | the standard benchmark; Kaggle disk is tight, Colab Pro / own GPU |

**Encoder** (`ENCODER`) and **how it starts** (`PRETRAINED`):

| `ENCODER` | Family | `PRETRAINED = True` loads |
|---|---|---|
| `"bia-vit"` | the project's own ViT | (always from scratch) |
| `"vit_b_16"`, `"vit_b_32"`, `"vit_l_16"` | torchvision ViT | ImageNet-1k weights |
| `"resnet18"` … `"resnet152"` | torchvision ResNet | ImageNet-1k weights |
| `"hf:google/vit-base-patch16-224-in21k"` | Hugging Face ViT | ImageNet-21k (14M images) |
| `"hf:facebook/dinov2-base"` | Hugging Face DINOv2 | self-supervised, 142M images |
| `"hf:openai/clip-vit-base-patch16"` | CLIP image tower | 400M image-text pairs |

`PRETRAINED = False` starts any of them from random weights. Pretrained
encoders get a small learning rate (1e-5), encoders trained from scratch the
decoder's (3e-4). `FREEZE_ENCODER = True` trains only the decoder: much faster,
slightly lower quality. Batch size and precision adapt to the GPU.

`SMOKE_TEST = True` runs a tiny version (few images, small model) to check the
whole notebook in a few minutes. Keep it `False` for real training.

In [ ]:
import dataclasses
from pathlib import Path

import torch

from bia_captioner.config import TrainConfig, data_config_for

SMOKE_TEST = False          # True = tiny quick check of the whole notebook
DATASET = "flowers"         # "shapes" | "flowers" | "birds" | "flickr8k" | "flickr30k" | "coco"
ENCODER = "vit_b_16"        # "bia-vit" | "vit_b_16" | "resnet50" | "hf:<model id>" ...
PRETRAINED = True           # False = train the encoder from scratch
FREEZE_ENCODER = False      # True = train only the decoder
USE_GOOGLE_DRIVE = False    # Colab only: keep checkpoints on Drive (survive disconnects)
RESUME = True               # continue from last.pt in OUTPUT_DIR if it exists

RUN_NAME = f"{DATASET}-{ENCODER.replace('hf:', '').replace('/', '-')}"
RUN_NAME += "-pretrained" if PRETRAINED and ENCODER != "bia-vit" else "-scratch"

# --- where checkpoints go ------------------------------------------------------
ON_KAGGLE = Path("/kaggle/working").exists()
ON_COLAB = "COLAB_RELEASE_TAG" in os.environ
if ON_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    OUTPUT_DIR = Path(f"/content/drive/MyDrive/bia-captioner/{RUN_NAME}")
elif ON_KAGGLE:
    OUTPUT_DIR = Path(f"/kaggle/working/checkpoints/{RUN_NAME}")
elif ON_COLAB:
    OUTPUT_DIR = Path(f"/content/checkpoints/{RUN_NAME}")
else:
    OUTPUT_DIR = Path(f"checkpoints/{RUN_NAME}")

# --- device, precision, batch size -----------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TARGET_BATCH = TrainConfig().batch_size  # 64: the effective batch we keep
if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    gpu_gb = props.total_memory / 1e9
    precision = "bf16" if props.major >= 8 else "fp16"  # native bf16 needs compute capability 8+
    batch_size = 64 if gpu_gb >= 20 or FREEZE_ENCODER else 32
    print(f"GPU: {props.name} ({gpu_gb:.0f} GB)")
else:
    precision, batch_size = "fp32", 16
    print("No GPU: enable one in the notebook settings (fine only for SMOKE_TEST / shapes).")

data_config = data_config_for(DATASET)
train_config = TrainConfig(
    output_dir=OUTPUT_DIR,
    batch_size=batch_size,
    grad_accum_steps=max(1, TARGET_BATCH // batch_size),
    freeze_encoder=FREEZE_ENCODER,
    precision=precision,
    num_workers=min(4, os.cpu_count() or 1),
    device=device.type,
    eval_every=500,
    log_every=25,
)
model_size = {"encoder": ENCODER, "pretrained": PRETRAINED}
if not PRETRAINED or ENCODER == "bia-vit":
    train_config = dataclasses.replace(train_config, epochs=40)  # from scratch: longer

if DATASET == "shapes":
    # Small images and a small model are enough for shapes (bia-vit: 8px patches).
    data_config = dataclasses.replace(data_config, image_size=64 if not PRETRAINED else 224)
    model_size.update(d_model=128, attention_heads=4, decoder_layers=2, encoder_layers=3,
                      patch_size=8)
    train_config = dataclasses.replace(train_config, epochs=8, lr=1e-3, warmup_steps=100,
                                       eval_every=250, bleu_samples=200)

if SMOKE_TEST:
    model_size.update(d_model=128, attention_heads=4, decoder_layers=2, encoder_layers=2,
                      dropout=0.0)
    OUTPUT_DIR = OUTPUT_DIR.parent / "smoke-test"  # never mixed with a real run
    train_config = dataclasses.replace(
        train_config, output_dir=OUTPUT_DIR, epochs=4, batch_size=8, grad_accum_steps=1,
        train_limit=64, val_limit=16, warmup_steps=2, log_every=2, eval_every=8,
        bleu_samples=16, lr=1e-3, num_workers=0,
    )
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if RESUME and (OUTPUT_DIR / T.LAST_CHECKPOINT).exists():
    train_config = dataclasses.replace(train_config, resume=OUTPUT_DIR / T.LAST_CHECKPOINT)

amp_dtype, _ = T.resolve_precision(train_config.precision, device)
for name, cfg in [("DataConfig", data_config), ("TrainConfig", train_config)]:
    print(f"\n{name}:")
    for key, value in dataclasses.asdict(cfg).items():
        print(f"  {key:18} {value}")
print(f"\nmodel: {model_size}")

## 4. Data

One item per image. Training sees every image once per epoch with a random one
of its captions; validation always uses the first caption, and BLEU compares
with all of them. Images are normalised the way the chosen encoder expects.

In [ ]:
import logging

import matplotlib.pyplot as plt

from bia_captioner.data import CaptionCollator, build_dataset
from bia_captioner.model import encoder_normalization
from bia_captioner.tokenizer import build_tokenizer

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s", datefmt="%H:%M:%S",
                    force=True)
for name in ("httpx", "httpx2"):  # hide Hugging Face HTTP logs
    logging.getLogger(name).setLevel(logging.WARNING)
T.seed_everything(train_config.seed)

tokenizer = build_tokenizer(data_config.tokenizer_id, data_config.cache_dir)
model_config = T.build_model_config(tokenizer, data_config, **model_size)
mean, std = encoder_normalization(model_config)
data_config = dataclasses.replace(data_config, image_mean=mean, image_std=std)

train_dataset = build_dataset(data_config, tokenizer, data_config.train_split,
                              limit=train_config.train_limit, train=train_config.augment)
val_dataset = build_dataset(data_config, tokenizer, data_config.val_split,
                            limit=train_config.val_limit)
print(f"train: {len(train_dataset):,} images | validation: {len(val_dataset):,} images")


def show_image(pixel_values, ax, title=""):
    """Undo the normalisation and draw the image."""
    m = torch.tensor(data_config.image_mean).view(3, 1, 1)
    s = torch.tensor(data_config.image_std).view(3, 1, 1)
    ax.imshow((pixel_values.cpu() * s + m).clamp(0, 1).permute(1, 2, 0))
    ax.set_title(title, fontsize=8, wrap=True)
    ax.axis("off")


fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for i, ax in enumerate(axes):
    item = val_dataset[i]
    show_image(item["pixel_values"], ax, val_dataset.references(i)[0])
plt.show()

In [ ]:
collator = CaptionCollator(pad_token_id=tokenizer.pad_token_id)
train_loader, val_loader = T.build_dataloaders(train_dataset, val_dataset, collator,
                                               train_config, device)
print(f"{len(train_loader):,} train batches per epoch")

## 5. Model

In [ ]:
import math

from bia_captioner.model import ImageCaptioner

model = ImageCaptioner(model_config, image_size=data_config.image_size).to(device)
n_params = sum(p.numel() for p in model.parameters())
n_encoder = sum(p.numel() for p in model.encoder.parameters())
start = "pretrained" if model_config.encoder_is_pretrained else "from scratch"
print(f"parameters: {n_params / 1e6:.1f}M (encoder {ENCODER}, {start}, {n_encoder / 1e6:.1f}M)")
print(f"image tokens per image: {model.num_image_tokens}")

# Sanity check: an untrained decoder should be close to random guessing, ln(vocab).
batch = next(iter(val_loader))
with torch.no_grad(), T.autocast(device, amp_dtype):
    small = T.move_batch({k: v[:4] for k, v in batch.items()}, device)
    _, loss = model(small["pixel_values"], small["input_ids"], labels=small["labels"])
print(f"initial loss {loss.item():.2f} (random guessing = {math.log(model_config.vocab_size):.2f})")

## 6. Training with a live dashboard

Same loop as `cap-train`: AdamW (smaller LR for a pretrained encoder), warmup +
cosine schedule, mixed precision, gradient accumulation and clipping, label
smoothing. Every `eval_every` steps it validates (loss, accuracy, BLEU on 500
validation images, one sample caption) and saves `last.pt`, plus `best.pt`
when the validation loss improves.

**Interrupted, or hit the session limit?** Run the notebook again with
`RESUME = True`: it continues from `last.pt` (same step, same data order).

In [ ]:
import json

from IPython.display import clear_output

METRICS_FILE = OUTPUT_DIR / "metrics.json"
history = {"train": [], "val": []}
if train_config.resume is not None and METRICS_FILE.exists():
    history = json.loads(METRICS_FILE.read_text())


def draw_dashboard():
    train, val = history["train"], history["val"]
    clear_output(wait=True)
    fig, axes = plt.subplots(2, 3, figsize=(16, 8))
    (ax_loss, ax_acc, ax_bleu), (ax_lr, ax_grad, ax_speed) = axes

    def series(rows, key):
        rows = [r for r in rows if key in r]
        return [r["step"] for r in rows], [r[key] for r in rows]

    ax_loss.plot(*series(train, "loss"), label="train", alpha=0.8)
    ax_loss.plot(*series(val, "loss"), "o-", label="validation")
    ax_loss.set_title("loss (lower is better)")
    ax_acc.plot(*series(train, "accuracy"), label="train", alpha=0.8)
    ax_acc.plot(*series(val, "accuracy"), "o-", label="validation")
    ax_acc.set_title("next-token accuracy")
    for n in (1, 4):
        ax_bleu.plot(*series(val, f"bleu{n}"), "o-", label=f"BLEU-{n}")
    ax_bleu.set_title("validation BLEU (greedy)")
    for group in ("decoder", "encoder"):
        ax_lr.plot(*series(train, f"lr_{group}"), label=group)
    ax_lr.set_title("learning rate")
    ax_lr.set_yscale("log")
    ax_grad.plot(*series(train, "grad_norm"), color="tab:red", alpha=0.8)
    ax_grad.axhline(train_config.grad_clip, ls="--", color="gray", label="clip")
    ax_grad.set_title("gradient norm (before clipping)")
    ax_speed.plot(*series(train, "images_per_sec"), color="tab:green")
    ax_speed.set_title("speed (images / second)")
    for ax in axes.flat:
        ax.set_xlabel("optimizer step")
        ax.grid(alpha=0.3)
        if ax.get_legend_handles_labels()[0]:
            ax.legend()
    fig.tight_layout()
    plt.show()
    if train:
        last = train[-1]
        print(f"step {last['step']:,}/{last['total_steps']:,} | epoch {last['epoch'] + 1} | "
              f"train loss {last['loss']:.4f} | {last['images_per_sec']:.0f} img/s")
    if val:
        last = val[-1]
        print(f"validation: loss {last['loss']:.4f} | BLEU-4 {100 * last.get('bleu4', 0):.2f}")
        print(f"  true:      {last['sample_target']}")
        print(f"  generated: {last['sample_prediction']}")


def on_metrics(event):
    row = {k: v for k, v in event.items() if k not in ("kind", "lr")}
    for group, lr in event.get("lr", {}).items():
        row[f"lr_{group}"] = lr
    history[event["kind"]].append(row)
    METRICS_FILE.write_text(json.dumps(history))
    draw_dashboard()


state = T.train(model, train_loader, val_loader, tokenizer, data_config, model_config,
                train_config, callback=on_metrics)
print(f"Training finished. Best validation loss {state.best_val_loss:.4f}")

## 7. Results

### 7.1 Captions for validation images (best checkpoint, beam search)

In [ ]:
from bia_captioner.generate import load_model

best_model, _ = load_model(OUTPUT_DIR / T.BEST_CHECKPOINT, device)
fig, axes = plt.subplots(2, 4, figsize=(16, 9))
for i, ax in enumerate(axes.flat):
    i = i % len(val_dataset)
    item = val_dataset[i]
    with torch.inference_mode(), T.autocast(device, amp_dtype):
        tokens, _ = best_model.beam_search(item["pixel_values"][None].to(device), beam_size=3)[0]
    caption = tokenizer.decode(tokens.tolist(), skip_special_tokens=True).strip()
    show_image(item["pixel_values"], ax, f"{caption}\n(true: {val_dataset.references(i)[0]})")
plt.tight_layout()
plt.show()

### 7.2 BLEU on the test split, and a shareable gallery

The test split was never used for training or model selection. Beam size 3.
`gallery.png` shows test photos with the model's captions (and one human
caption in grey): ready for a README or a post.

In [ ]:
from bia_captioner import evaluate

test_args = ["--checkpoint", str(OUTPUT_DIR / T.BEST_CHECKPOINT), "--split", "test",
             "--beam-size", "3", "--output", str(OUTPUT_DIR / "test_predictions.json"),
             "--gallery", str(OUTPUT_DIR / "gallery.png")]
if SMOKE_TEST:
    test_args += ["--limit", "16"]
test_scores = evaluate.run(test_args)

from IPython.display import Image as ShowImage, display

display(ShowImage(filename=str(OUTPUT_DIR / "gallery.png")))  # ready to share

## 8. Download the trained model

Zips `best.pt`, the tokenizer, the metrics and the test predictions
(`last.pt` is left out: it is only needed to resume). Use it with the repo:

```bash
unzip bia-captioner-run.zip -d checkpoints/run
cap-generate --checkpoint checkpoints/run/best.pt --image photo.jpg
```

In [ ]:
import shutil
import tempfile

with tempfile.TemporaryDirectory() as tmp:
    bundle = Path(tmp) / "bundle"
    bundle.mkdir()
    for name in [T.BEST_CHECKPOINT, "metrics.json", "test_predictions.json", "gallery.png"]:
        if (OUTPUT_DIR / name).exists():
            shutil.copy(OUTPUT_DIR / name, bundle / name)
    shutil.copytree(OUTPUT_DIR / "tokenizer", bundle / "tokenizer")
    archive = shutil.make_archive(str(OUTPUT_DIR.parent / "bia-captioner-run"), "zip", bundle)
print("created", archive)

if ON_COLAB:
    from google.colab import files

    files.download(archive)
elif ON_KAGGLE:
    print("Kaggle: click 'Save Version', then download it from the notebook's Output tab.")